In [78]:
# Import required libraries
import pandas as pd
import os
import glob

In [79]:
# Define base path and years
BASE_PATH = "../data/raw"
YEARS = list(range(2015, 2023))

# Define table types and their categories
TABLE_TYPES = {
    "Table_1.2": "Total Accidental Deaths",
    "Table_1A.4": "Mode of Transportation",
    "Table_1A.5": "Month of Occurrence",
    "Table_1A.6": "Time of Occurrence",
    "Table_1A.7": "Road Classification",
    "Table_1A.11": "Place of Occurrence"
}

In [80]:
# Load all files and store in a dictionary
all_files = {}

for year in YEARS:
    all_files[year] = {}
    for table, category in TABLE_TYPES.items():
        file_path = os.path.join(BASE_PATH, str(year), f"ADSI_{year}_{table}.csv")
        if os.path.exists(file_path):
            try:
                df = pd.read_csv(file_path, encoding='utf-8')
                all_files[year][table] = {"df": df, "status": "loaded", "category": category}
            except UnicodeDecodeError:
                try:
                    df = pd.read_csv(file_path, encoding='latin1')
                    all_files[year][table] = {"df": df, "status": "loaded (latin1)", "category": category}
                except Exception as e:
                    all_files[year][table] = {"df": None, "status": f"error: {e}", "category": category}
        else:
            all_files[year][table] = {"df": None, "status": "file not found", "category": category}

print("All files loading attempt complete.")

All files loading attempt complete.


In [81]:
# Inspect one file per table type (using 2020 as reference)
REFERENCE_YEAR = 2020

for table, category in TABLE_TYPES.items():
    print(f"\n{'='*60}")
    print(f"Category : {category}")
    print(f"Table    : {table} | Year: {REFERENCE_YEAR}")
    print(f"{'='*60}")
    
    entry = all_files[REFERENCE_YEAR][table]
    if entry["df"] is not None:
        df = entry["df"]
        print(f"Shape    : {df.shape}")
        print(f"Columns  : {list(df.columns)}")
        print(f"\nFirst 5 rows:")
        print(df.head())
    else:
        print(f"Status   : {entry['status']}")


Category : Total Accidental Deaths
Table    : Table_1.2 | Year: 2020
Shape    : (93, 9)
Columns  : ['Si. No. (Col. 1)', 'Category', 'State/UT/City (Col. 2)', 'Total Number of Accidental Deaths - Forces of Nature (Col. 3)', 'Total Number of Accidental Deaths - Other Causes (Col. 4)', 'Total Number of Accidental Deaths - Total (Col. 5)', 'Percentage Share in Total Deaths (Col. 6)', 'Projected Mid-Year Population (in Lakh) (Col. 7)', 'Rate of Accidental Deaths (Col. 8) = (Col.5/ Col.7)']

First 5 rows:
  Si. No. (Col. 1) Category State/UT/City (Col. 2)  \
0                1    State         Andhra Pradesh   
1                2    State      Arunachal Pradesh   
2                3    State                  Assam   
3                4    State                  Bihar   
4                5    State           Chhattisgarh   

   Total Number of Accidental Deaths - Forces of Nature (Col. 3)  \
0                                                164               
1                                

In [82]:
# Standard State/UT names mapping
STATE_UT_NAMES = {
    # States
    "ANDHRA PRADESH": "Andhra Pradesh",
    "ARUNACHAL PRADESH": "Arunachal Pradesh",
    "ASSAM": "Assam",
    "BIHAR": "Bihar",
    "CHHATTISGARH": "Chhattisgarh",
    "GOA": "Goa",
    "GUJARAT": "Gujarat",
    "HARYANA": "Haryana",
    "HIMACHAL PRADESH": "Himachal Pradesh",
    "JHARKHAND": "Jharkhand",
    "KARNATAKA": "Karnataka",
    "KERALA": "Kerala",
    "MADHYA PRADESH": "Madhya Pradesh",
    "MAHARASHTRA": "Maharashtra",
    "MANIPUR": "Manipur",
    "MEGHALAYA": "Meghalaya",
    "MIZORAM": "Mizoram",
    "NAGALAND": "Nagaland",
    "ODISHA": "Odisha",
    "PUNJAB": "Punjab",
    "RAJASTHAN": "Rajasthan",
    "SIKKIM": "Sikkim",
    "TAMIL NADU": "Tamil Nadu",
    "TELANGANA": "Telangana",
    "TRIPURA": "Tripura",
    "UTTAR PRADESH": "Uttar Pradesh",
    "UTTARAKHAND": "Uttarakhand",
    "WEST BENGAL": "West Bengal",
    # UTs
    "ANDAMAN AND NICOBAR ISLANDS": "Andaman and Nicobar Islands",
    "ANDAMAN & NICOBAR ISLANDS": "Andaman and Nicobar Islands",
    "CHANDIGARH": "Chandigarh",
    "DADRA AND NAGAR HAVELI": "Dadra and Nagar Haveli and Daman and Diu",
    "DAMAN AND DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "DADRA AND NAGAR HAVELI AND DAMAN AND DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "DADRA & NAGAR HAVELI & DAMAN & DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "D & N HAVELI AND DAMAN & DIU": "Dadra and Nagar Haveli and Daman and Diu",
"ANDAMAND AND NICOBAR ISLANDS": "Andaman and Nicobar Islands",
    "DELHI": "Delhi",
    "JAMMU AND KASHMIR": "Jammu and Kashmir",
    "JAMMU & KASHMIR": "Jammu and Kashmir",
    "LADAKH": "Jammu and Kashmir",
    "LAKSHADWEEP": "Lakshadweep",
    "PUDUCHERRY": "Puducherry",
    "PONDICHERRY": "Puducherry",
}
print("State/UT name mapping defined.")

State/UT name mapping defined.


In [83]:
# Clean Table_1.2 - Total Accidental Deaths
all_years_data = []

for year in YEARS:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    
    # Find category column and name column
    cat_col = df.columns[1]
    name_col = df.columns[2]
    
    # 2019 has different structure — category col has state names instead of State/UT label
    if df[cat_col].str.strip().str.upper().isin(["STATE", "UT", "UNION TERRITORY"]).any():
        # Normal structure — 2015, 2016, 2017, 2018, 2020, 2021, 2022
        df = df[df[cat_col].str.strip().str.upper().isin(["STATE", "UT", "UNION TERRITORY"])]
    else:
        # 2019 structure — filter by known State/UT names
        df = df[df[name_col].str.strip().str.upper().isin(VALID_STATE_UT_KEYS)]
        # Swap: name is in col 2 (index 1) actually
        name_col = df.columns[1]

    # Find total deaths column dynamically
    total_col = None
    for col in df.columns:
        if "Total" in col and ("Col. 5" in col or "Col.5" in col or "Col. 6" in col):
            total_col = col
            break
    # Fallback — last numeric-looking column with Total
    if total_col is None:
        for col in df.columns:
            if "Total" in col and "Number" in col:
                total_col = col
                break
    # Fallback 2 — for 2017 style
    if total_col is None:
        for col in df.columns:
            if "Total" in col and "Death" in col:
                total_col = col
                break

    if total_col is None:
        print(f"WARNING: Total column not found for year {year}")
        print(f"Columns: {list(df.columns)}")
        continue

    # Extract relevant columns
    df_clean = df[[name_col, total_col]].copy()
    df_clean.columns = ["State/UT", "Total Accidental Deaths"]

    # Standardize State/UT names
    df_clean["State/UT"] = df_clean["State/UT"].str.strip().str.upper().map(STATE_UT_NAMES)

    # Drop unmapped rows
    df_clean = df_clean.dropna(subset=["State/UT"])

    # Add year column
    df_clean["Year"] = year

    # Convert to numeric
    df_clean["Total Accidental Deaths"] = pd.to_numeric(df_clean["Total Accidental Deaths"], errors="coerce")

    # Merge Dadra & Daman and J&K & Ladakh if separate
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False)["Total Accidental Deaths"].sum()

    all_years_data.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 29 States/UTs processed.
2016 — 32 States/UTs processed.
2017 — 32 States/UTs processed.
2018 — 32 States/UTs processed.


AttributeError: Can only use .str accessor with string values, not integer

In [84]:
# Debug - check 2017 and 2019 structure
for year in [2017, 2019]:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    print(f"\n{year} — Columns: {list(df.columns[:4])}")
    print(f"{year} — First 5 rows of first 4 columns:")
    print(df.iloc[:5, :4])
    print(f"{year} — Unique values in column 2 (Category): {df.iloc[:, 1].unique()}")


2017 — Columns: ['Sl. No.', 'Category', 'State/UT/City', 'Deaths due to Forces of Nature - 2016']
2017 — First 5 rows of first 4 columns:
  Sl. No. Category      State/UT/City  Deaths due to Forces of Nature - 2016
0       1    State     Andhra Pradesh                                    426
1       2    State  Arunachal Pradesh                                     47
2       3    State              Assam                                    544
3       4    State              Bihar                                   1057
4       5    State       Chhattisgarh                                    305
2017 — Unique values in column 2 (Category): <StringArray>
['State', 'Union Territory', 'Total (All India)', 'City']
Length: 4, dtype: str

2019 — Columns: ['Category (Col. 1)', 'State/UT/City (Col. 2)', 'Number of Accidental Deaths - Forces of Nature (Col. 3)', 'Number of Accidental Deaths - Other Causes (Col. 4)']
2019 — First 5 rows of first 4 columns:
  Category (Col. 1) State/UT/City (Col. 2

In [85]:
# Reload 2017 file and check structure of 2017 and 2019
for year in [2017, 2019]:
    file_path = os.path.join(BASE_PATH, str(year), f"ADSI_{year}_Table_1.2.csv")
    df = pd.read_csv(file_path, encoding='utf-8')
    print(f"\n{year} — Columns: {list(df.columns[:4])}")
    print(f"{year} — First 5 rows of first 4 columns:")
    print(df.iloc[:5, :4])
    print(f"{year} — Unique values in column 1 (Category): {df.iloc[:, 1].unique()}")


2017 — Columns: ['Sl. No.', 'Category', 'State/UT/City', 'Deaths due to Forces of Nature - 2016']
2017 — First 5 rows of first 4 columns:
  Sl. No. Category      State/UT/City  Deaths due to Forces of Nature - 2016
0       1    State     Andhra Pradesh                                    426
1       2    State  Arunachal Pradesh                                     47
2       3    State              Assam                                    544
3       4    State              Bihar                                   1057
4       5    State       Chhattisgarh                                    305
2017 — Unique values in column 1 (Category): <StringArray>
['State', 'Union Territory', 'Total (All India)', 'City']
Length: 4, dtype: str

2019 — Columns: ['Category (Col. 1)', 'State/UT/City (Col. 2)', 'Number of Accidental Deaths - Forces of Nature (Col. 3)', 'Number of Accidental Deaths - Other Causes (Col. 4)']
2019 — First 5 rows of first 4 columns:
  Category (Col. 1) State/UT/City (Col. 2

In [86]:
# Standard State/UT names mapping
STATE_UT_NAMES = {
    # States
    "ANDHRA PRADESH": "Andhra Pradesh",
    "ARUNACHAL PRADESH": "Arunachal Pradesh",
    "ASSAM": "Assam",
    "BIHAR": "Bihar",
    "CHHATTISGARH": "Chhattisgarh",
    "GOA": "Goa",
    "GUJARAT": "Gujarat",
    "HARYANA": "Haryana",
    "HIMACHAL PRADESH": "Himachal Pradesh",
    "JHARKHAND": "Jharkhand",
    "KARNATAKA": "Karnataka",
    "KERALA": "Kerala",
    "MADHYA PRADESH": "Madhya Pradesh",
    "MAHARASHTRA": "Maharashtra",
    "MANIPUR": "Manipur",
    "MEGHALAYA": "Meghalaya",
    "MIZORAM": "Mizoram",
    "NAGALAND": "Nagaland",
    "ODISHA": "Odisha",
    "PUNJAB": "Punjab",
    "RAJASTHAN": "Rajasthan",
    "SIKKIM": "Sikkim",
    "TAMIL NADU": "Tamil Nadu",
    "TELANGANA": "Telangana",
    "TRIPURA": "Tripura",
    "UTTAR PRADESH": "Uttar Pradesh",
    "UTTARAKHAND": "Uttarakhand",
    "WEST BENGAL": "West Bengal",
    # UTs - full forms
    "ANDAMAN AND NICOBAR ISLANDS": "Andaman and Nicobar Islands",
    "ANDAMAN & NICOBAR ISLANDS": "Andaman and Nicobar Islands",
    "A & N ISLANDS": "Andaman and Nicobar Islands",
    "A & NISLANDS": "Andaman and Nicobar Islands",
    "CHANDIGARH": "Chandigarh",
    "DADRA AND NAGAR HAVELI": "Dadra and Nagar Haveli and Daman and Diu",
    "DAMAN AND DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "DADRA AND NAGAR HAVELI AND DAMAN AND DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "DADRA & NAGAR HAVELI & DAMAN & DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "D & N HAVELI": "Dadra and Nagar Haveli and Daman and Diu",
    "DAMAN & DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "DELHI": "Delhi",
    "DELHI (UT)": "Delhi",
    "JAMMU AND KASHMIR": "Jammu and Kashmir",
    "JAMMU & KASHMIR": "Jammu and Kashmir",
    "LADAKH": "Jammu and Kashmir",
    "LAKSHADWEEP": "Lakshadweep",
    "PUDUCHERRY": "Puducherry",
    "PONDICHERRY": "Puducherry",
}

# Valid State/UT names list for 2019 type filtering
VALID_STATE_UT_KEYS = set(STATE_UT_NAMES.keys())

print("State/UT name mapping defined.")

State/UT name mapping defined.


In [87]:
# Debug 2017 and 2019
for year in [2017, 2019]:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    cat_col = df.columns[1]
    name_col = df.columns[2]
    print(f"\n{year}")
    print(f"cat_col: {cat_col}")
    print(f"name_col: {name_col}")
    print(f"cat_col unique values: {df[cat_col].unique()[:5]}")
    print(f"name_col dtype: {df[name_col].dtype}")
    print(df.iloc[:3, :4])


2017
cat_col: Category
name_col: State/UT/City
cat_col unique values: <StringArray>
['State', 'Union Territory', 'Total (All India)', 'City']
Length: 4, dtype: str
name_col dtype: str
  Sl. No. Category      State/UT/City  Deaths due to Forces of Nature - 2016
0       1    State     Andhra Pradesh                                    426
1       2    State  Arunachal Pradesh                                     47
2       3    State              Assam                                    544

2019
cat_col: State/UT/City (Col. 2)
name_col: Number of Accidental Deaths - Forces of Nature (Col. 3)
cat_col unique values: <StringArray>
['ANDHRA PRADESH', 'ARUNACHAL PRADESH', 'ASSAM', 'BIHAR', 'CHHATTISGARH']
Length: 5, dtype: str
name_col dtype: int64
  Category (Col. 1) State/UT/City (Col. 2)  \
0             State         ANDHRA PRADESH   
1             State      ARUNACHAL PRADESH   
2             State                  ASSAM   

   Number of Accidental Deaths - Forces of Nature (Col. 3)  \
0

In [88]:
# Fix 1 — Reload 2017 file
file_path_2017 = os.path.join(BASE_PATH, "2017", "ADSI_2017_Table_1.2.csv")
df_2017 = pd.read_csv(file_path_2017, encoding='utf-8')
all_files[2017]["Table_1.2"]["df"] = df_2017
print(f"2017 reloaded — Unique Category values: {df_2017.iloc[:, 1].unique()}")

# Fix 2 — Check 2019 column positions
df_2019 = all_files[2019]["Table_1.2"]["df"].copy()
df_2019.columns = df_2019.columns.str.strip()
print(f"\n2019 — col[0]: {df_2019.columns[0]}")
print(f"2019 — col[1]: {df_2019.columns[1]}")
print(f"2019 — col[0] unique (first 5): {df_2019.iloc[:, 0].unique()[:5]}")

2017 reloaded — Unique Category values: <StringArray>
['State', 'Union Territory', 'Total (All India)', 'City']
Length: 4, dtype: str

2019 — col[0]: Category (Col. 1)
2019 — col[1]: State/UT/City (Col. 2)
2019 — col[0] unique (first 5): <StringArray>
['State', 'UT', 'TOTAL (ALL INDIA)', 'City']
Length: 4, dtype: str


In [89]:
# Clean Table_1.2 - Total Accidental Deaths
all_years_data = []

for year in YEARS:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()

    # Detect column positions — some years have category in col[0], others in col[1]
    if df.iloc[:, 0].str.strip().str.upper().isin(["STATE", "UT", "UNION TERRITORY"]).any():
        cat_col = df.columns[0]
        name_col = df.columns[1]
    else:
        cat_col = df.columns[1]
        name_col = df.columns[2]

    # Filter only State and UT rows
    df = df[df[cat_col].str.strip().str.upper().isin(["STATE", "UT", "UNION TERRITORY"])]

    # Find total deaths column dynamically
    total_col = None
    for col in df.columns:
        if "Total" in col and ("Col. 5" in col or "Col.5" in col or "Col. 6" in col):
            total_col = col
            break
    if total_col is None:
        for col in df.columns:
            if "Total" in col and "Number" in col:
                total_col = col
                break
    if total_col is None:
        for col in df.columns:
            if "Total" in col and "Death" in col:
                total_col = col
                break

    if total_col is None:
        print(f"WARNING: Total column not found for year {year}")
        print(f"Columns: {list(df.columns)}")
        continue

    # Extract relevant columns
    df_clean = df[[name_col, total_col]].copy()
    df_clean.columns = ["State/UT", "Total Accidental Deaths"]

    # Standardize State/UT names
    df_clean["State/UT"] = df_clean["State/UT"].str.strip().str.upper().map(STATE_UT_NAMES)

    # Drop unmapped rows
    df_clean = df_clean.dropna(subset=["State/UT"])

    # Add year column
    df_clean["Year"] = year

    # Convert to numeric
    df_clean["Total Accidental Deaths"] = pd.to_numeric(df_clean["Total Accidental Deaths"], errors="coerce")

    # Merge Dadra & Daman and J&K & Ladakh if separate
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False)["Total Accidental Deaths"].sum()

    all_years_data.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 29 States/UTs processed.
2016 — 0 States/UTs processed.
2017 — 0 States/UTs processed.
2018 — 0 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 28 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 28 States/UTs processed.

All years processed!


In [90]:
# Debug column positions for all years
for year in YEARS:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    print(f"\n{year}")
    print(f"col[0]: {df.columns[0]} — unique (first 3): {df.iloc[:, 0].unique()[:3]}")
    print(f"col[1]: {df.columns[1]} — unique (first 3): {df.iloc[:, 1].unique()[:3]}")


2015
col[0]: Sl. No. (Col.1) — unique (first 3): <StringArray>
['1', '2', '3']
Length: 3, dtype: str
col[1]: Category (Col.2) — unique (first 3): <StringArray>
['State', 'TOTAL (STATES)', 'Union Territories']
Length: 3, dtype: str

2016
col[0]: Sl. No. (Col.1) — unique (first 3): <StringArray>
['1', '2', '3']
Length: 3, dtype: str
col[1]: Category (Col.2) — unique (first 3): <StringArray>
['State', 'Union Territory', 'Total (All India)']
Length: 3, dtype: str

2017
col[0]: Sl. No. — unique (first 3): <StringArray>
['1', '2', '3']
Length: 3, dtype: str
col[1]: Category — unique (first 3): <StringArray>
['State', 'Union Territory', 'Total (All India)']
Length: 3, dtype: str

2018
col[0]: Sl. No. - (Col.1) — unique (first 3): <StringArray>
['1', '2', '3']
Length: 3, dtype: str
col[1]: Category - (Col.2) — unique (first 3): <StringArray>
['State', 'Union Territory', 'Total (All India)']
Length: 3, dtype: str

2019
col[0]: Category (Col. 1) — unique (first 3): <StringArray>
['State', 'UT',

In [91]:
# Clean Table_1.2 - Total Accidental Deaths
all_years_data = []

# Category values that mean State or UT
VALID_CATEGORIES = ["STATE", "UT", "UNION TERRITORY", "UNION TERRITORIES"]

for year in YEARS:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()

    # Detect column positions
    if df.iloc[:, 0].str.strip().str.upper().isin(VALID_CATEGORIES).any():
        # 2019, 2021 type
        cat_col = df.columns[0]
        name_col = df.columns[1]
    else:
        # 2015, 2016, 2017, 2018, 2020, 2022 type
        cat_col = df.columns[1]
        name_col = df.columns[2]

    # Filter only State and UT rows
    df = df[df[cat_col].str.strip().str.upper().isin(VALID_CATEGORIES)]

    # Find total deaths column dynamically
    total_col = None
    for col in df.columns:
        if "Total" in col and ("Col. 5" in col or "Col.5" in col or "Col. 6" in col):
            total_col = col
            break
    if total_col is None:
        for col in df.columns:
            if "Total" in col and "Number" in col:
                total_col = col
                break
    if total_col is None:
        for col in df.columns:
            if "Total" in col and "Death" in col:
                total_col = col
                break

    if total_col is None:
        print(f"WARNING: Total column not found for year {year}")
        print(f"Columns: {list(df.columns)}")
        continue

    # Extract relevant columns
    df_clean = df[[name_col, total_col]].copy()
    df_clean.columns = ["State/UT", "Total Accidental Deaths"]

    # Standardize State/UT names
    df_clean["State/UT"] = df_clean["State/UT"].str.strip().str.upper().map(STATE_UT_NAMES)

    # Drop unmapped rows
    df_clean = df_clean.dropna(subset=["State/UT"])

    # Add year column
    df_clean["Year"] = year

    # Convert to numeric
    df_clean["Total Accidental Deaths"] = pd.to_numeric(df_clean["Total Accidental Deaths"], errors="coerce")

    # Merge Dadra & Daman and J&K & Ladakh if separate
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False)["Total Accidental Deaths"].sum()

    all_years_data.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 35 States/UTs processed.
2016 — 0 States/UTs processed.
2017 — 0 States/UTs processed.
2018 — 0 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 34 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 34 States/UTs processed.

All years processed!


In [92]:
# Debug total column for 2016, 2017, 2018
for year in [2016, 2017, 2018]:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    print(f"\n{year} — All columns:")
    for col in df.columns:
        print(f"  {col}")


2016 — All columns:
  Sl. No. (Col.1)
  Category (Col.2)
  State/UT (Col.3)
  Total Number of Accidental Deaths - Forces of Nature (Col.4)
  Total Number of Accidental Deaths - Other Causes (Col.5)
  Total Number of Accidental Deaths - Total (Col.6)
  Percentage Share in Total Deaths (Col.7)
  Projected Mid-Year Population* (In Lakh+) (Col.8)
  Rate of Accidental Deaths (Col.9) = (Col.6/Col.8)

2017 — All columns:
  Sl. No.
  Category
  State/UT/City
  Deaths due to Forces of Nature - 2016
  Deaths due to Forces of Nature - 2017
  Deaths due to Forces of Nature - % Variation
  Deaths due to Other Causes - 2016
  Deaths due to Other Causes - 2017
  Deaths due to Other Causes - % Variation
  Total Accidental Deaths - 2016
  Total Accidental Deaths - 2017
  Total Accidental Deaths - % Variation

2018 — All columns:
  Sl. No. - (Col.1)
  Category - (Col.2)
  State/UT - (Col.3)
  Total Number of Accidental Deaths - Forces of Nature - (Col.4)
  Total Number of Accidental Deaths - Other Caus

In [93]:
# Clean Table_1.2 - Total Accidental Deaths
all_years_data = []

VALID_CATEGORIES = ["STATE", "UT", "UNION TERRITORY", "UNION TERRITORIES"]

def find_total_col(df, year):
    for col in df.columns:
        # 2017 specific
        if f"Total Accidental Deaths - {year}" in col:
            return col
    for col in df.columns:
        if "Total" in col and ("Col. 5" in col or "Col.5" in col):
            return col
    for col in df.columns:
        if "Total" in col and ("Col. 6" in col or "Col.6" in col):
            return col
    for col in df.columns:
        if "Total" in col and "Number" in col:
            return col
    for col in df.columns:
        if "Total" in col and "Death" in col:
            return col
    return None

for year in YEARS:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()

    # Detect column positions
    if df.iloc[:, 0].str.strip().str.upper().isin(VALID_CATEGORIES).any():
        cat_col = df.columns[0]
        name_col = df.columns[1]
    else:
        cat_col = df.columns[1]
        name_col = df.columns[2]

    # Filter only State and UT rows
    df = df[df[cat_col].str.strip().str.upper().isin(VALID_CATEGORIES)]

    # Find total deaths column
    total_col = find_total_col(df, year)

    if total_col is None:
        print(f"WARNING: Total column not found for year {year}")
        continue

    # Extract relevant columns
    df_clean = df[[name_col, total_col]].copy()
    df_clean.columns = ["State/UT", "Total Accidental Deaths"]

    # Standardize State/UT names
    df_clean["State/UT"] = df_clean["State/UT"].str.strip().str.upper().map(STATE_UT_NAMES)

    # Drop unmapped rows
    df_clean = df_clean.dropna(subset=["State/UT"])

    # Add year column
    df_clean["Year"] = year

    # Convert to numeric
    df_clean["Total Accidental Deaths"] = pd.to_numeric(df_clean["Total Accidental Deaths"], errors="coerce")

    # Merge Dadra & Daman and J&K & Ladakh if separate
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False)["Total Accidental Deaths"].sum()

    all_years_data.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 35 States/UTs processed.
2016 — 0 States/UTs processed.
2017 — 0 States/UTs processed.
2018 — 0 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 34 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 34 States/UTs processed.

All years processed!


In [94]:
# Debug 2016, 2017, 2018 step by step
for year in [2016, 2017, 2018]:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    
    # Check column detection
    if df.iloc[:, 0].str.strip().str.upper().isin(VALID_CATEGORIES).any():
        cat_col = df.columns[0]
        name_col = df.columns[1]
    else:
        cat_col = df.columns[1]
        name_col = df.columns[2]
    
    print(f"\n{year}")
    print(f"cat_col: {cat_col}")
    print(f"name_col: {name_col}")
    
    # Check after filter
    df_filtered = df[df[cat_col].str.strip().str.upper().isin(VALID_CATEGORIES)]
    print(f"Rows after filter: {len(df_filtered)}")
    print(f"Unique cat values: {df[cat_col].str.strip().str.upper().unique()[:5]}")
    
    # Check total col
    total_col = find_total_col(df_filtered, year)
    print(f"total_col found: {total_col}")
    
    # Check name mapping
    if len(df_filtered) > 0:
        sample = df_filtered[name_col].str.strip().str.upper().head(3)
        print(f"Sample names: {list(sample)}")


2016
cat_col: Sl. No. (Col.1)
name_col: Category (Col.2)
Rows after filter: 2
Unique cat values: <StringArray>
['1', '2', '3', '4', '5']
Length: 5, dtype: str
total_col found: Total Number of Accidental Deaths - Other Causes (Col.5)
Sample names: ['STATE', 'UNION TERRITORY']

2017
cat_col: Sl. No.
name_col: Category
Rows after filter: 2
Unique cat values: <StringArray>
['1', '2', '3', '4', '5']
Length: 5, dtype: str
total_col found: Total Accidental Deaths - 2017
Sample names: ['STATE', 'UNION TERRITORY']

2018
cat_col: Sl. No. - (Col.1)
name_col: Category - (Col.2)
Rows after filter: 2
Unique cat values: <StringArray>
['1', '2', '3', '4', '5']
Length: 5, dtype: str
total_col found: Total Number of Accidental Deaths - Other Causes - (Col.5)
Sample names: ['STATE', 'UNION TERRITORY']


In [95]:
# Clean Table_1.2 - Total Accidental Deaths
all_years_data = []

VALID_CATEGORIES = ["STATE", "UT", "UNION TERRITORY", "UNION TERRITORIES"]

def find_total_col(df, year):
    # 2017 specific
    for col in df.columns:
        if f"Total Accidental Deaths - {year}" in col:
            return col
    # Col.6 priority
    for col in df.columns:
        if "Total" in col and ("Col. 6" in col or "Col.6" in col):
            return col
    # Col.5 only if nothing else
    for col in df.columns:
        if "Total" in col and ("Col. 5" in col or "Col.5" in col):
            return col
    for col in df.columns:
        if "Total" in col and "Number" in col:
            return col
    for col in df.columns:
        if "Total" in col and "Death" in col:
            return col
    return None

def detect_cols(df):
    # Check col[1] first — most years have Category there
    if df.iloc[:, 1].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES).any():
        return df.columns[1], df.columns[2]
    # Check col[0] — 2019, 2021 type
    elif df.iloc[:, 0].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES).any():
        return df.columns[0], df.columns[1]
    else:
        return df.columns[1], df.columns[2]

for year in YEARS:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()

    cat_col, name_col = detect_cols(df)

    # Filter only State and UT rows
    df = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]

    # Find total deaths column
    total_col = find_total_col(df, year)

    if total_col is None:
        print(f"WARNING: Total column not found for year {year}")
        continue

    # Extract relevant columns
    df_clean = df[[name_col, total_col]].copy()
    df_clean.columns = ["State/UT", "Total Accidental Deaths"]

    # Standardize State/UT names
    df_clean["State/UT"] = df_clean["State/UT"].astype(str).str.strip().str.upper().map(STATE_UT_NAMES)

    # Drop unmapped rows
    df_clean = df_clean.dropna(subset=["State/UT"])

    # Add year column
    df_clean["Year"] = year

    # Convert to numeric
    df_clean["Total Accidental Deaths"] = pd.to_numeric(df_clean["Total Accidental Deaths"], errors="coerce")

    # Merge Dadra & Daman and J&K & Ladakh if separate
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False)["Total Accidental Deaths"].sum()

    all_years_data.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 35 States/UTs processed.
2016 — 35 States/UTs processed.
2017 — 35 States/UTs processed.
2018 — 35 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 34 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 34 States/UTs processed.

All years processed!


In [96]:
# Debug 2020 and 2022 — which State/UT is missing
all_35 = set(all_years_data[0]["State/UT"].unique())  # 2015 as reference

for i, year in enumerate(YEARS):
    if len(all_years_data[i]) == 34:
        present = set(all_years_data[i]["State/UT"].unique())
        missing = all_35 - present
        print(f"{year} — Missing: {missing}")

2020 — Missing: {'Dadra and Nagar Haveli and Daman and Diu'}
2022 — Missing: {'Andaman and Nicobar Islands'}


In [97]:
# Debug 2020 and 2022 — check raw names
for year in [2020, 2022]:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    cat_col, name_col = detect_cols(df)
    df_filtered = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    print(f"\n{year} — Raw State/UT names:")
    print(df_filtered[name_col].str.strip().tolist())


2020 — Raw State/UT names:
['Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chhattisgarh', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal', 'A & N Islands', 'Chandigarh', 'D & N Haveli and Daman & Diu', 'Delhi (UT)', 'Jammu & Kashmir', 'Ladakh', 'Lakshadweep', 'Puducherry']

2022 — Raw State/UT names:
['Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chhattisgarh', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal', 'Andamand and Nicobar Islands', 'Chandigarh', 'Dadra and Nagar Haveli and D

In [98]:
# Debug — check which States/UTs are missing for each year
reference = set(['Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chhattisgarh', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal', 'Andaman and Nicobar Islands', 'Chandigarh', 'Dadra and Nagar Haveli and Daman and Diu', 'Delhi', 'Jammu and Kashmir', 'Lakshadweep', 'Puducherry'])

for i, year in enumerate(YEARS):
    present = set(all_years_data[i]["State/UT"].unique())
    missing = reference - present
    extra = present - reference
    print(f"{year} — Missing: {missing}")
    print(f"{year} — Extra: {extra}")
    print()

2015 — Missing: set()
2015 — Extra: set()

2016 — Missing: set()
2016 — Extra: set()

2017 — Missing: set()
2017 — Extra: set()

2018 — Missing: set()
2018 — Extra: set()

2019 — Missing: set()
2019 — Extra: set()

2020 — Missing: {'Dadra and Nagar Haveli and Daman and Diu'}
2020 — Extra: set()

2021 — Missing: set()
2021 — Extra: set()

2022 — Missing: {'Andaman and Nicobar Islands'}
2022 — Extra: set()



In [99]:
# Debug — check raw names for missing entries in 2015, 2019, 2020
search_terms = ["DELHI", "ANDAMAN", "DADRA", "DAMAN", "HAVELI"]

for year in [2015, 2019, 2020]:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    cat_col, name_col = detect_cols(df)
    df_filtered = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    names = df_filtered[name_col].astype(str).str.strip().tolist()
    matched = [n for n in names if any(term in n.upper() for term in search_terms)]
    print(f"\n{year} — Relevant raw names: {matched}")


2015 — Relevant raw names: ['D & N HAVELI', 'DAMAN & DIU', 'DELHI (UT)']

2019 — Relevant raw names: ['D & N HAVELI', 'DAMAN & DIU', 'DELHI (UT)']

2020 — Relevant raw names: ['D & N Haveli and Daman & Diu', 'Delhi (UT)']


In [100]:
# Debug — check Andaman raw names for 2015-2019
for year in [2015, 2016, 2017, 2018, 2019]:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    cat_col, name_col = detect_cols(df)
    df_filtered = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    names = df_filtered[name_col].astype(str).str.strip().tolist()
    matched = [n for n in names if "ANDAMAN" in n.upper() or "A &" in n.upper()]
    print(f"{year} — Andaman raw name: {matched}")

2015 — Andaman raw name: ['A & N ISLANDS']
2016 — Andaman raw name: ['A & N Islands']
2017 — Andaman raw name: ['A & N Islands']
2018 — Andaman raw name: ['A & N Islands']
2019 — Andaman raw name: ['A & NISLANDS']


In [101]:
# Updated STATE_UT_NAMES mapping
STATE_UT_NAMES = {
    # States
    "ANDHRA PRADESH": "Andhra Pradesh",
    "ARUNACHAL PRADESH": "Arunachal Pradesh",
    "ASSAM": "Assam",
    "BIHAR": "Bihar",
    "CHHATTISGARH": "Chhattisgarh",
    "GOA": "Goa",
    "GUJARAT": "Gujarat",
    "HARYANA": "Haryana",
    "HIMACHAL PRADESH": "Himachal Pradesh",
    "JHARKHAND": "Jharkhand",
    "KARNATAKA": "Karnataka",
    "KERALA": "Kerala",
    "MADHYA PRADESH": "Madhya Pradesh",
    "MAHARASHTRA": "Maharashtra",
    "MANIPUR": "Manipur",
    "MEGHALAYA": "Meghalaya",
    "MIZORAM": "Mizoram",
    "NAGALAND": "Nagaland",
    "ODISHA": "Odisha",
    "PUNJAB": "Punjab",
    "RAJASTHAN": "Rajasthan",
    "SIKKIM": "Sikkim",
    "TAMIL NADU": "Tamil Nadu",
    "TELANGANA": "Telangana",
    "TRIPURA": "Tripura",
    "UTTAR PRADESH": "Uttar Pradesh",
    "UTTARAKHAND": "Uttarakhand",
    "WEST BENGAL": "West Bengal",
    # UTs
    "ANDAMAN AND NICOBAR ISLANDS": "Andaman and Nicobar Islands",
    "ANDAMAN & NICOBAR ISLANDS": "Andaman and Nicobar Islands",
    "A & N ISLANDS": "Andaman and Nicobar Islands",
    "A & NISLANDS": "Andaman and Nicobar Islands",
    "ANDAMAND AND NICOBAR ISLANDS": "Andaman and Nicobar Islands",
    "CHANDIGARH": "Chandigarh",
    "DADRA AND NAGAR HAVELI": "Dadra and Nagar Haveli and Daman and Diu",
    "DAMAN AND DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "DADRA AND NAGAR HAVELI AND DAMAN AND DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "DADRA & NAGAR HAVELI & DAMAN & DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "D & N HAVELI": "Dadra and Nagar Haveli and Daman and Diu",
    "DAMAN & DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "D & N HAVELI AND DAMAN & DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "DELHI": "Delhi",
    "DELHI (UT)": "Delhi",
    "JAMMU AND KASHMIR": "Jammu and Kashmir",
    "JAMMU & KASHMIR": "Jammu and Kashmir",
    "LADAKH": "Jammu and Kashmir",
    "LAKSHADWEEP": "Lakshadweep",
    "PUDUCHERRY": "Puducherry",
    "PONDICHERRY": "Puducherry",
}

VALID_STATE_UT_KEYS = set(STATE_UT_NAMES.keys())
print("State/UT name mapping updated.")

State/UT name mapping updated.


In [102]:
# Merge all years and add S.No.
final_df = pd.concat(all_years_data, ignore_index=True)

# Sort by Year then alphabetically by State/UT
final_df = final_df.sort_values(["Year", "State/UT"]).reset_index(drop=True)

# Add continuous S.No.
final_df.insert(0, "S.No.", range(1, len(final_df) + 1))

# Reorder columns
final_df = final_df[["S.No.", "State/UT", "Year", "Total Accidental Deaths"]]

# Save to processed folder
final_df.to_csv("../data/processed/total_accidental_deaths_2015_2022.csv", index=False)

print(f"Total rows: {len(final_df)}")
print(f"Years: {final_df['Year'].unique()}")
print(f"\nFirst 10 rows:")
print(final_df.head(10))

Total rows: 278
Years: [2015 2016 2017 2018 2019 2020 2021 2022]

First 10 rows:
   S.No.                                  State/UT  Year  \
0      1               Andaman and Nicobar Islands  2015   
1      2                            Andhra Pradesh  2015   
2      3                         Arunachal Pradesh  2015   
3      4                                     Assam  2015   
4      5                                     Bihar  2015   
5      6                                Chandigarh  2015   
6      7                              Chhattisgarh  2015   
7      8  Dadra and Nagar Haveli and Daman and Diu  2015   
8      9                                     Delhi  2015   
9     10                                       Goa  2015   

   Total Accidental Deaths  
0                     91.0  
1                   1321.0  
2                     23.0  
3                    185.0  
4                    481.0  
5                     67.0  
6                   4679.0  
7                     42.0

In [103]:
# Inspect Table_1A.4 structure for all years
for year in YEARS:
    df = all_files[year]["Table_1A.4"]["df"].copy()
    df.columns = df.columns.str.strip()
    print(f"\n{year} — Shape: {df.shape}")
    print(f"Columns:")
    for col in df.columns:
        print(f"  {col}")


2015 — Shape: (93, 48)
Columns:
  Sl. No
  Category
  State/UT/City
  Truck/Lorry - Offenders
  Truck/Lorry - Victims
  Truck/Lorry - Total
  Bus - Offenders
  Bus - Victims
  Bus - Total
  SUV/Station Wagon/etc. - Offenders
  SUV/Station Wagon/etc. - Victims
  SUV/Station Wagon/etc. - Total
  Car - Offenders
  Car - Victims
  Car - Total
  Jeep - Offenders
  Jeep - Victims
  Jeep - Total
  Tractor - Offenders
  Tractor - Victims
  Tractor - Total
  Three Wheeler/Auto Rickshaw - Offenders
  Three Wheeler/Auto Rickshaw - Victims
  Three Wheeler/Auto Rickshaw - Total
  Two Wheeler - Offenders
  Two Wheeler - Victims
  Two Wheeler - Total
  Other Motor Vehicles - Offenders
  Other Motor Vehicles - Victims
  Other Motor Vehicles - Total
  Bicycle - Offenders
  Bicycle - Victims
  Bicycle - Total
  Hand Drawn Vehicle/Cycle Rickshaw - Offenders
  Hand Drawn Vehicle/Cycle Rickshaw - Victims
  Hand Drawn Vehicle/Cycle Rickshaw - Total
  Animal Drawn Vehicle - Offenders
  Animal Drawn Vehicle 

In [104]:
# Reload 2021 Table_1A.4
file_path_2021 = os.path.join(BASE_PATH, "2021", "ADSI_2021_Table_1A.4.csv")
df_2021 = pd.read_csv(file_path_2021, encoding='utf-8')
all_files[2021]["Table_1A.4"]["df"] = df_2021
print(f"2021 reloaded — Shape: {df_2021.shape}")
print(f"Columns:")
for col in df_2021.columns:
    print(f"  {col}")

2021 reloaded — Shape: (93, 28)
Columns:
  Category
  State/UT/City
  Truck/Lorry/Mini Truck - Injured
  Truck/Lorry/Mini Truck - Died
  Bus - Injured
  Bus - Died
  SUV / Car / Jeep / etc. - Injured
  SUV / Car / Jeep / etc. - Died
  Tractor - Injured
  Tractor - Died
  Three Wheeler/Auto Rickshaw - Injured
  Three Wheeler/Auto Rickshaw - Died
  Two Wheeler - Injured
  Two Wheeler - Died
  Other Motor Vehicles - Injured
  Other Motor Vehicles - Died
  Bicycle - Injured
  Bicycle - Died
  Hand Drawn Vehicle/ Cycle Rickshaw - Injured
  Hand Drawn Vehicle/Cycle Rickshaw - Died
  Animal Drawn Vehicle - Injured
  Animal Drawn Vehicle - Died
  Others - Injured
  Others - Died
  Pedestrian - Injured
  Pedestrian - Died
  Grand Total - Injured
  Grand Total - Died


In [105]:
# Clean Table_1A.4 - Mode of Transportation
all_years_transport = []

# Vehicle column mapping for each year type
# Type A: 2015-2020 — uses " - Total" suffix
# Type B: 2021-2022 — uses " - Died" suffix

VEHICLE_MAPPING_A = {
    "Truck/Lorry/Mini Truck - Died": ["Truck/Lorry - Total", "Truck/Lorry/Mini Truck - Total"],
    "Bus - Died": ["Bus - Total"],
    "SUV/Car/Jeep - Died": ["SUV/Station Wagon/etc. - Total", "Car - Total", "Jeep - Total"],
    "Tractor - Died": ["Tractor - Total"],
    "Three Wheeler/Auto Rickshaw - Died": ["Three Wheeler/Auto Rickshaw - Total"],
    "Two Wheeler - Died": ["Two Wheeler - Total"],
    "Other Motor Vehicles - Died": [
        "Other Motor Vehicles - Total",
        "Bicycle - Total",
        "Hand Drawn Vehicle/Cycle Rickshaw - Total",
        "Hand Drawn Vehicle/ Cycle Rickshaw - Total",
        "Animal Drawn Vehicle - Total",
        "Pedestrian - Total",
        "Others - Total"
    ]
}

VEHICLE_MAPPING_B = {
    "Truck/Lorry/Mini Truck - Died": ["Truck/Lorry/Mini Truck - Died"],
    "Bus - Died": ["Bus - Died"],
    "SUV/Car/Jeep - Died": ["SUV / Car / Jeep / etc. - Died"],
    "Tractor - Died": ["Tractor - Died"],
    "Three Wheeler/Auto Rickshaw - Died": ["Three Wheeler/Auto Rickshaw - Died"],
    "Two Wheeler - Died": ["Two Wheeler - Died"],
    "Other Motor Vehicles - Died": [
        "Other Motor Vehicles - Died",
        "Bicycle - Died",
        "Hand Drawn Vehicle/ Cycle Rickshaw - Died",
        "Hand Drawn Vehicle/Cycle Rickshaw - Died",
        "Animal Drawn Vehicle - Died",
        "Pedestrian - Died",
        "Others - Died"
    ]
}

for year in YEARS:
    df = all_files[year]["Table_1A.4"]["df"].copy()
    df.columns = df.columns.str.strip()

    # Detect column positions
    if df.iloc[:, 0].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES).any():
        cat_col = df.columns[0]
        name_col = df.columns[1]
    else:
        cat_col = df.columns[1]
        name_col = df.columns[2]

    # Filter only State and UT rows
    df = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]

    # Select mapping based on year
    mapping = VEHICLE_MAPPING_A if year <= 2020 else VEHICLE_MAPPING_B

    # Build cleaned dataframe
    df_clean = df[[name_col]].copy()
    df_clean.columns = ["State/UT"]

    for new_col, source_cols in mapping.items():
        # Find available source columns
        available = [c for c in source_cols if c in df.columns]
        if available:
            df_clean[new_col] = df[available].apply(pd.to_numeric, errors="coerce").sum(axis=1)
        else:
            df_clean[new_col] = 0
            print(f"WARNING: {year} — No source columns found for {new_col}")

    # Standardize State/UT names
    df_clean["State/UT"] = df_clean["State/UT"].astype(str).str.strip().str.upper().map(STATE_UT_NAMES)

    # Drop unmapped rows
    df_clean = df_clean.dropna(subset=["State/UT"])

    # Add year
    df_clean["Year"] = year

    # Merge Dadra & Daman and J&K & Ladakh if separate
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False).sum()

    all_years_transport.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 35 States/UTs processed.
2016 — 0 States/UTs processed.
2017 — 0 States/UTs processed.
2018 — 0 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 35 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 0 States/UTs processed.

All years processed!


In [106]:
# Debug 2016, 2017, 2018, 2019, 2022
for year in [2016, 2017, 2018, 2019, 2022]:
    df = all_files[year]["Table_1A.4"]["df"].copy()
    df.columns = df.columns.str.strip()
    print(f"\n{year}")
    print(f"col[0]: {df.columns[0]} — unique (first 3): {df.iloc[:, 0].unique()[:3]}")
    print(f"col[1]: {df.columns[1]} — unique (first 3): {df.iloc[:, 1].unique()[:3]}")


2016
col[0]: Sl. No. — unique (first 3): <StringArray>
['1', '2', '3']
Length: 3, dtype: str
col[1]: Category — unique (first 3): <StringArray>
['State', 'Union Territory', 'Total (All India)']
Length: 3, dtype: str

2017
col[0]: Sl. No. — unique (first 3): <StringArray>
['1', '2', '3']
Length: 3, dtype: str
col[1]: Category — unique (first 3): <StringArray>
['State', 'Union Territory', 'Total (All India)']
Length: 3, dtype: str

2018
col[0]: Sl. No. — unique (first 3): <StringArray>
['1', '2', '3']
Length: 3, dtype: str
col[1]: Category — unique (first 3): <StringArray>
['State', 'Union Territory', 'Total (All India)']
Length: 3, dtype: str

2019
col[0]: Category — unique (first 3): <StringArray>
['State', 'UT', 'TOTAL (ALL INDIA)']
Length: 3, dtype: str
col[1]: State/UT/City — unique (first 3): <StringArray>
['ANDHRA PRADESH', 'ARUNACHAL PRADESH', 'ASSAM']
Length: 3, dtype: str

2022
col[0]: Sl. No. — unique (first 3): <StringArray>
['1', '2', '3']
Length: 3, dtype: str
col[1]: Stat

In [107]:
# Debug name mapping for 2016, 2017, 2018 and column names for 2019, 2022
for year in [2016, 2017, 2018]:
    df = all_files[year]["Table_1A.4"]["df"].copy()
    df.columns = df.columns.str.strip()
    cat_col, name_col = detect_cols(df)
    df_filtered = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    print(f"\n{year} — cat_col: {cat_col}, name_col: {name_col}")
    print(f"Rows after filter: {len(df_filtered)}")
    print(f"Sample names: {df_filtered[name_col].head(3).tolist()}")

# Check 2019 column names with underscore
print(f"\n2019 — Sample columns:")
df_2019 = all_files[2019]["Table_1A.4"]["df"].copy()
df_2019.columns = df_2019.columns.str.strip()
print([c for c in df_2019.columns if "Truck" in c or "Bus" in c])

# Check 2022 — no category column
print(f"\n2022 — col[1] sample values:")
df_2022 = all_files[2022]["Table_1A.4"]["df"].copy()
df_2022.columns = df_2022.columns.str.strip()
print(df_2022.iloc[:5, 1].tolist())


2016 — cat_col: Category, name_col: State/UT/City
Rows after filter: 38
Sample names: ['Andhra Pradesh', 'Arunachal Pradesh', 'Assam']

2017 — cat_col: Category, name_col: State/UT/City
Rows after filter: 38
Sample names: ['Andhra Pradesh', 'Arunachal Pradesh', 'Assam']

2018 — cat_col: Category, name_col: State/UT/City
Rows after filter: 38
Sample names: ['Andhra Pradesh', 'Arunachal Pradesh', 'Assam']

2019 — Sample columns:
['Truck/Lorry_Offenders', 'Truck/Lorry_Victims', 'Truck/Lorry_Total', 'Bus_Offenders', 'Bus_Victims', 'Bus_Total']

2022 — col[1] sample values:
['Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chhattisgarh']


In [108]:
# Clean Table_1A.4 - Mode of Transportation
all_years_transport = []

VEHICLE_MAPPING_A = {
    "Truck/Lorry/Mini Truck - Died": ["Truck/Lorry - Total", "Truck/Lorry/Mini Truck - Total"],
    "Bus - Died": ["Bus - Total"],
    "SUV/Car/Jeep - Died": ["SUV/Station Wagon/etc. - Total", "Car - Total", "Jeep - Total"],
    "Tractor - Died": ["Tractor - Total"],
    "Three Wheeler/Auto Rickshaw - Died": ["Three Wheeler/Auto Rickshaw - Total"],
    "Two Wheeler - Died": ["Two Wheeler - Total"],
    "Other Motor Vehicles - Died": [
        "Other Motor Vehicles - Total",
        "Bicycle - Total",
        "Hand Drawn Vehicle/Cycle Rickshaw - Total",
        "Hand Drawn Vehicle/ Cycle Rickshaw - Total",
        "Animal Drawn Vehicle - Total",
        "Pedestrian - Total",
        "Others - Total"
    ]
}

VEHICLE_MAPPING_2019 = {
    "Truck/Lorry/Mini Truck - Died": ["Truck/Lorry_Total"],
    "Bus - Died": ["Bus_Total"],
    "SUV/Car/Jeep - Died": ["SUV/Station Wagon/etc._Total", "Car_Total", "Jeep_Total"],
    "Tractor - Died": ["Tractor_Total"],
    "Three Wheeler/Auto Rickshaw - Died": ["Three Wheeler/Auto Rickshaw_Total"],
    "Two Wheeler - Died": ["Two Wheeler_Total"],
    "Other Motor Vehicles - Died": [
        "Other Motor Vehicles_Total",
        "Bicycle_Total",
        "Hand Drawn Vehicle/ Cycle Rickshaw_Total",
        "Hand Drawn Vehicle/Cycle Rickshaw_Total",
        "Animal Drawn Vehicle_Total",
        "Pedestrian_Total",
        "Others_Total"
    ]
}

VEHICLE_MAPPING_B = {
    "Truck/Lorry/Mini Truck - Died": ["Truck/Lorry/Mini Truck - Died"],
    "Bus - Died": ["Bus - Died"],
    "SUV/Car/Jeep - Died": ["SUV / Car / Jeep / etc. - Died"],
    "Tractor - Died": ["Tractor - Died"],
    "Three Wheeler/Auto Rickshaw - Died": ["Three Wheeler/Auto Rickshaw - Died"],
    "Two Wheeler - Died": ["Two Wheeler - Died"],
    "Other Motor Vehicles - Died": [
        "Other Motor Vehicles - Died",
        "Bicycle - Died",
        "Hand Drawn Vehicle/ Cycle Rickshaw - Died",
        "Hand Drawn Vehicle/Cycle Rickshaw - Died",
        "Animal Drawn Vehicle - Died",
        "Pedestrian - Died",
        "Others - Died"
    ]
}

for year in YEARS:
    df = all_files[year]["Table_1A.4"]["df"].copy()
    df.columns = df.columns.str.strip()

    # Detect column positions and filter
    if year == 2022:
        # No category column — filter by known State/UT names
        name_col = df.columns[1]
        df = df[df[name_col].astype(str).str.strip().str.upper().isin(VALID_STATE_UT_KEYS)]
    elif df.iloc[:, 0].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES).any():
        cat_col = df.columns[0]
        name_col = df.columns[1]
        df = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    else:
        cat_col = df.columns[1]
        name_col = df.columns[2]
        df = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]

    # Select mapping based on year
    if year == 2019:
        mapping = VEHICLE_MAPPING_2019
    elif year >= 2021:
        mapping = VEHICLE_MAPPING_B
    else:
        mapping = VEHICLE_MAPPING_A

    # Build cleaned dataframe
    df_clean = df[[name_col]].copy()
    df_clean.columns = ["State/UT"]

    for new_col, source_cols in mapping.items():
        available = [c for c in source_cols if c in df.columns]
        if available:
            df_clean[new_col] = df[available].apply(pd.to_numeric, errors="coerce").sum(axis=1)
        else:
            df_clean[new_col] = 0
            print(f"WARNING: {year} — No source columns found for {new_col}")

    # Standardize State/UT names
    df_clean["State/UT"] = df_clean["State/UT"].astype(str).str.strip().str.upper().map(STATE_UT_NAMES)

    # Drop unmapped rows
    df_clean = df_clean.dropna(subset=["State/UT"])

    # Add year
    df_clean["Year"] = year

    # Merge Dadra & Daman and J&K & Ladakh if separate
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False).sum()

    all_years_transport.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 35 States/UTs processed.
2016 — 0 States/UTs processed.
2017 — 0 States/UTs processed.
2018 — 0 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 35 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 35 States/UTs processed.

All years processed!


In [109]:
# Debug 2016, 2017, 2018
for year in [2016, 2017, 2018]:
    df = all_files[year]["Table_1A.4"]["df"].copy()
    df.columns = df.columns.str.strip()
    
    cat_col = df.columns[1]
    name_col = df.columns[2]
    
    df_filtered = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    print(f"\n{year} — Rows after filter: {len(df_filtered)}")
    print(f"Sample raw names: {df_filtered[name_col].head(5).tolist()}")
    
    # Check mapping
    mapped = df_filtered[name_col].astype(str).str.strip().str.upper().map(STATE_UT_NAMES)
    print(f"Unmapped names: {df_filtered[name_col][mapped.isna()].tolist()}")


2016 — Rows after filter: 38
Sample raw names: ['Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chhattisgarh']
Unmapped names: ['Total (States)', 'Total (UTs)']

2017 — Rows after filter: 38
Sample raw names: ['Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chhattisgarh']
Unmapped names: ['Total (States)', 'Total (UTs)']

2018 — Rows after filter: 38
Sample raw names: ['Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chhattisgarh']
Unmapped names: ['Total (States)', 'Total (UTs)']


In [110]:
# Clean Table_1A.4 - Mode of Transportation
all_years_transport = []

VEHICLE_MAPPING_A = {
    "Truck/Lorry/Mini Truck - Died": ["Truck/Lorry - Total", "Truck/Lorry/Mini Truck - Total"],
    "Bus - Died": ["Bus - Total"],
    "SUV/Car/Jeep - Died": ["SUV/Station Wagon/etc. - Total", "Car - Total", "Jeep - Total"],
    "Tractor - Died": ["Tractor - Total"],
    "Three Wheeler/Auto Rickshaw - Died": ["Three Wheeler/Auto Rickshaw - Total"],
    "Two Wheeler - Died": ["Two Wheeler - Total"],
    "Other Motor Vehicles - Died": [
        "Other Motor Vehicles - Total",
        "Bicycle - Total",
        "Hand Drawn Vehicle/Cycle Rickshaw - Total",
        "Hand Drawn Vehicle/ Cycle Rickshaw - Total",
        "Animal Drawn Vehicle - Total",
        "Pedestrian - Total",
        "Others - Total"
    ]
}

VEHICLE_MAPPING_2019 = {
    "Truck/Lorry/Mini Truck - Died": ["Truck/Lorry_Total"],
    "Bus - Died": ["Bus_Total"],
    "SUV/Car/Jeep - Died": ["SUV/Station Wagon/etc._Total", "Car_Total", "Jeep_Total"],
    "Tractor - Died": ["Tractor_Total"],
    "Three Wheeler/Auto Rickshaw - Died": ["Three Wheeler/Auto Rickshaw_Total"],
    "Two Wheeler - Died": ["Two Wheeler_Total"],
    "Other Motor Vehicles - Died": [
        "Other Motor Vehicles_Total",
        "Bicycle_Total",
        "Hand Drawn Vehicle/ Cycle Rickshaw_Total",
        "Hand Drawn Vehicle/Cycle Rickshaw_Total",
        "Animal Drawn Vehicle_Total",
        "Pedestrian_Total",
        "Others_Total"
    ]
}

VEHICLE_MAPPING_B = {
    "Truck/Lorry/Mini Truck - Died": ["Truck/Lorry/Mini Truck - Died"],
    "Bus - Died": ["Bus - Died"],
    "SUV/Car/Jeep - Died": ["SUV / Car / Jeep / etc. - Died"],
    "Tractor - Died": ["Tractor - Died"],
    "Three Wheeler/Auto Rickshaw - Died": ["Three Wheeler/Auto Rickshaw - Died"],
    "Two Wheeler - Died": ["Two Wheeler - Died"],
    "Other Motor Vehicles - Died": [
        "Other Motor Vehicles - Died",
        "Bicycle - Died",
        "Hand Drawn Vehicle/ Cycle Rickshaw - Died",
        "Hand Drawn Vehicle/Cycle Rickshaw - Died",
        "Animal Drawn Vehicle - Died",
        "Pedestrian - Died",
        "Others - Died"
    ]
}

for year in YEARS:
    df = all_files[year]["Table_1A.4"]["df"].copy()
    df.columns = df.columns.str.strip()

    # Detect column positions and filter
    if year == 2022:
        name_col = df.columns[1]
        df = df[df[name_col].astype(str).str.strip().str.upper().isin(VALID_STATE_UT_KEYS)]
    elif df.iloc[:, 0].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES).any():
        cat_col = df.columns[0]
        name_col = df.columns[1]
        df = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    else:
        cat_col = df.columns[1]
        name_col = df.columns[2]
        df = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]

    # Remove subtotal rows like "Total (States)", "Total (UTs)"
    df = df[~df[name_col].astype(str).str.strip().str.upper().str.startswith("TOTAL")]

    # Select mapping based on year
    if year == 2019:
        mapping = VEHICLE_MAPPING_2019
    elif year >= 2021:
        mapping = VEHICLE_MAPPING_B
    else:
        mapping = VEHICLE_MAPPING_A

    # Build cleaned dataframe
    df_clean = df[[name_col]].copy()
    df_clean.columns = ["State/UT"]

    for new_col, source_cols in mapping.items():
        available = [c for c in source_cols if c in df.columns]
        if available:
            df_clean[new_col] = df[available].apply(pd.to_numeric, errors="coerce").sum(axis=1)
        else:
            df_clean[new_col] = 0
            print(f"WARNING: {year} — No source columns found for {new_col}")

    # Standardize State/UT names
    df_clean["State/UT"] = df_clean["State/UT"].astype(str).str.strip().str.upper().map(STATE_UT_NAMES)

    # Drop unmapped rows
    df_clean = df_clean.dropna(subset=["State/UT"])

    # Add year
    df_clean["Year"] = year

    # Merge Dadra & Daman and J&K & Ladakh if separate
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False).sum()

    all_years_transport.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 35 States/UTs processed.
2016 — 0 States/UTs processed.
2017 — 0 States/UTs processed.
2018 — 0 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 35 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 35 States/UTs processed.

All years processed!


In [111]:
# Targeted debug for 2016
year = 2016
df = all_files[year]["Table_1A.4"]["df"].copy()
df.columns = df.columns.str.strip()

print(f"col[0]: {df.columns[0]}")
print(f"col[1]: {df.columns[1]}")
print(f"col[2]: {df.columns[2]}")

# Simulate the filter
cat_col = df.columns[1]
name_col = df.columns[2]

df_filtered = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
print(f"\nRows after category filter: {len(df_filtered)}")

df_filtered = df_filtered[~df_filtered[name_col].astype(str).str.strip().str.upper().str.startswith("TOTAL")]
print(f"Rows after total filter: {len(df_filtered)}")

# Check mapping
names = df_filtered[name_col].astype(str).str.strip().str.upper()
mapped = names.map(STATE_UT_NAMES)
print(f"\nUnmapped names: {df_filtered[name_col][mapped.isna()].tolist()}")
print(f"Mapped count: {mapped.notna().sum()}")

col[0]: Sl. No.
col[1]: Category
col[2]: State/UT/City

Rows after category filter: 38
Rows after total filter: 36

Unmapped names: []
Mapped count: 36


In [112]:
# Step by step debug inside loop for 2016
year = 2016
df = all_files[year]["Table_1A.4"]["df"].copy()
df.columns = df.columns.str.strip()

print(f"col[0] isin VALID_CATEGORIES: {df.iloc[:, 0].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES).any()}")

if year == 2022:
    name_col = df.columns[1]
    df = df[df[name_col].astype(str).str.strip().str.upper().isin(VALID_STATE_UT_KEYS)]
elif df.iloc[:, 0].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES).any():
    cat_col = df.columns[0]
    name_col = df.columns[1]
    df = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    print(f"Branch 1 — cat_col: {cat_col}, name_col: {name_col}")
else:
    cat_col = df.columns[1]
    name_col = df.columns[2]
    df = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    print(f"Branch 2 — cat_col: {cat_col}, name_col: {name_col}")

print(f"Rows after filter: {len(df)}")
df = df[~df[name_col].astype(str).str.strip().str.upper().str.startswith("TOTAL")]
print(f"Rows after total removal: {len(df)}")

df_clean = df[[name_col]].copy()
df_clean.columns = ["State/UT"]
df_clean["State/UT"] = df_clean["State/UT"].astype(str).str.strip().str.upper().map(STATE_UT_NAMES)
df_clean = df_clean.dropna(subset=["State/UT"])
print(f"Rows after mapping: {len(df_clean)}")

col[0] isin VALID_CATEGORIES: True
Branch 1 — cat_col: Sl. No., name_col: Category
Rows after filter: 2
Rows after total removal: 2
Rows after mapping: 0


In [113]:
# Clean Table_1A.4 - Mode of Transportation
all_years_transport = []

VEHICLE_MAPPING_A = {
    "Truck/Lorry/Mini Truck - Died": ["Truck/Lorry - Total", "Truck/Lorry/Mini Truck - Total"],
    "Bus - Died": ["Bus - Total"],
    "SUV/Car/Jeep - Died": ["SUV/Station Wagon/etc. - Total", "Car - Total", "Jeep - Total"],
    "Tractor - Died": ["Tractor - Total"],
    "Three Wheeler/Auto Rickshaw - Died": ["Three Wheeler/Auto Rickshaw - Total"],
    "Two Wheeler - Died": ["Two Wheeler - Total"],
    "Other Motor Vehicles - Died": [
        "Other Motor Vehicles - Total",
        "Bicycle - Total",
        "Hand Drawn Vehicle/Cycle Rickshaw - Total",
        "Hand Drawn Vehicle/ Cycle Rickshaw - Total",
        "Animal Drawn Vehicle - Total",
        "Pedestrian - Total",
        "Others - Total"
    ]
}

VEHICLE_MAPPING_2019 = {
    "Truck/Lorry/Mini Truck - Died": ["Truck/Lorry_Total"],
    "Bus - Died": ["Bus_Total"],
    "SUV/Car/Jeep - Died": ["SUV/Station Wagon/etc._Total", "Car_Total", "Jeep_Total"],
    "Tractor - Died": ["Tractor_Total"],
    "Three Wheeler/Auto Rickshaw - Died": ["Three Wheeler/Auto Rickshaw_Total"],
    "Two Wheeler - Died": ["Two Wheeler_Total"],
    "Other Motor Vehicles - Died": [
        "Other Motor Vehicles_Total",
        "Bicycle_Total",
        "Hand Drawn Vehicle/ Cycle Rickshaw_Total",
        "Hand Drawn Vehicle/Cycle Rickshaw_Total",
        "Animal Drawn Vehicle_Total",
        "Pedestrian_Total",
        "Others_Total"
    ]
}

VEHICLE_MAPPING_B = {
    "Truck/Lorry/Mini Truck - Died": ["Truck/Lorry/Mini Truck - Died"],
    "Bus - Died": ["Bus - Died"],
    "SUV/Car/Jeep - Died": ["SUV / Car / Jeep / etc. - Died"],
    "Tractor - Died": ["Tractor - Died"],
    "Three Wheeler/Auto Rickshaw - Died": ["Three Wheeler/Auto Rickshaw - Died"],
    "Two Wheeler - Died": ["Two Wheeler - Died"],
    "Other Motor Vehicles - Died": [
        "Other Motor Vehicles - Died",
        "Bicycle - Died",
        "Hand Drawn Vehicle/ Cycle Rickshaw - Died",
        "Hand Drawn Vehicle/Cycle Rickshaw - Died",
        "Animal Drawn Vehicle - Died",
        "Pedestrian - Died",
        "Others - Died"
    ]
}

def detect_transport_cols(df, year):
    # 2022 has no category column
    if year == 2022:
        return None, df.columns[1]
    # Check col[1] first — most years have Category there
    if df.iloc[:, 1].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES).any():
        return df.columns[1], df.columns[2]
    # Check col[0] — 2019, 2021 type
    elif df.iloc[:, 0].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES).any():
        return df.columns[0], df.columns[1]
    else:
        return df.columns[1], df.columns[2]

for year in YEARS:
    df = all_files[year]["Table_1A.4"]["df"].copy()
    df.columns = df.columns.str.strip()

    cat_col, name_col = detect_transport_cols(df, year)

    # Filter only State and UT rows
    if cat_col is not None:
        df = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    else:
        df = df[df[name_col].astype(str).str.strip().str.upper().isin(VALID_STATE_UT_KEYS)]

    # Remove subtotal rows
    df = df[~df[name_col].astype(str).str.strip().str.upper().str.startswith("TOTAL")]

    # Select mapping based on year
    if year == 2019:
        mapping = VEHICLE_MAPPING_2019
    elif year >= 2021:
        mapping = VEHICLE_MAPPING_B
    else:
        mapping = VEHICLE_MAPPING_A

    # Build cleaned dataframe
    df_clean = df[[name_col]].copy()
    df_clean.columns = ["State/UT"]

    for new_col, source_cols in mapping.items():
        available = [c for c in source_cols if c in df.columns]
        if available:
            df_clean[new_col] = df[available].apply(pd.to_numeric, errors="coerce").sum(axis=1)
        else:
            df_clean[new_col] = 0
            print(f"WARNING: {year} — No source columns found for {new_col}")

    # Standardize State/UT names
    df_clean["State/UT"] = df_clean["State/UT"].astype(str).str.strip().str.upper().map(STATE_UT_NAMES)

    # Drop unmapped rows
    df_clean = df_clean.dropna(subset=["State/UT"])

    # Add year
    df_clean["Year"] = year

    # Merge Dadra & Daman and J&K & Ladakh if separate
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False).sum()

    all_years_transport.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 35 States/UTs processed.
2016 — 35 States/UTs processed.
2017 — 35 States/UTs processed.
2018 — 35 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 35 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 35 States/UTs processed.

All years processed!


In [114]:
# Merge all years and add S.No.
final_transport_df = pd.concat(all_years_transport, ignore_index=True)

# Sort by Year then alphabetically by State/UT
final_transport_df = final_transport_df.sort_values(["Year", "State/UT"]).reset_index(drop=True)

# Add continuous S.No.
final_transport_df.insert(0, "S.No.", range(1, len(final_transport_df) + 1))

# Reorder columns
final_transport_df = final_transport_df[["S.No.", "State/UT", "Year",
                                          "Truck/Lorry/Mini Truck - Died",
                                          "Bus - Died",
                                          "SUV/Car/Jeep - Died",
                                          "Tractor - Died",
                                          "Three Wheeler/Auto Rickshaw - Died",
                                          "Two Wheeler - Died",
                                          "Other Motor Vehicles - Died"]]

# Save to processed folder
final_transport_df.to_csv("../data/processed/mode_of_transport_2015_2022.csv", index=False)

print(f"Total rows: {len(final_transport_df)}")
print(f"Years: {final_transport_df['Year'].unique()}")
print(f"\nFirst 5 rows:")
print(final_transport_df.head())

Total rows: 280
Years: [2015 2016 2017 2018 2019 2020 2021 2022]

First 5 rows:
   S.No.                     State/UT  Year  Truck/Lorry/Mini Truck - Died  \
0      1  Andaman and Nicobar Islands  2015                              6   
1      2               Andhra Pradesh  2015                           1389   
2      3            Arunachal Pradesh  2015                             45   
3      4                        Assam  2015                            360   
4      5                        Bihar  2015                           1135   

   Bus - Died  SUV/Car/Jeep - Died  Tractor - Died  \
0           3                    3               1   
1         593                 1394             384   
2           2                   61               7   
3         446                  660              57   
4         757                  837             300   

   Three Wheeler/Auto Rickshaw - Died  Two Wheeler - Died  \
0                                   1                   9   
1   

In [115]:
# Inspect Table_1A.5 structure for all years
for year in YEARS:
    df = all_files[year]["Table_1A.5"]["df"].copy()
    df.columns = df.columns.str.strip()
    print(f"\n{year} — Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")


2015 — Shape: (93, 55)
Columns: ['Sl. No.', 'Category', 'State/UT/City', 'Road Accidents - Jan.', 'Road Accidents - Feb.', 'Road Accidents - Mar.', 'Road Accidents - Apr.', 'Road Accidents - May', 'Road Accidents - Jun.', 'Road Accidents - Jul.', 'Road Accidents - Aug.', 'Road Accidents - Sep.', 'Road Accidents - Oct.', 'Road Accidents - Nov.', 'Road Accidents - Dec.', 'Road Accidents - Total', 'Railway Crossing Accidents - Jan.', 'Railway Crossing Accidents - Feb.', 'Railway Crossing Accidents - Mar.', 'Railway Crossing Accidents - Apr.', 'Railway Crossing Accidents - May', 'Railway Crossing Accidents - Jun.', 'Railway Crossing Accidents - Jul.', 'Railway Crossing Accidents - Aug.', 'Railway Crossing Accidents - Sep.', 'Railway Crossing Accidents - Oct.', 'Railway Crossing Accidents - Nov.', 'Railway Crossing Accidents - Dec.', 'Railway Crossing Accidents - Total', 'Railway Accidents - Jan.', 'Railway Accidents - Feb.', 'Railway Accidents - Mar.', 'Railway Accidents - Apr.', 'Railway

In [116]:
# Reload 2016 Table_1A.5
file_path_2016 = os.path.join(BASE_PATH, "2016", "ADSI_2016_Table_1A.5.csv")
df_2016 = pd.read_csv(file_path_2016, encoding='utf-8')
all_files[2016]["Table_1A.5"]["df"] = df_2016
print(f"2016 reloaded — Shape: {df_2016.shape}")
print(f"Columns: {list(df_2016.columns)}")

2016 reloaded — Shape: (93, 55)
Columns: ['Sl. No.', 'Category', 'State/UT/CITY', 'Road Accidents - Jan.', 'Road Accidents - Feb.', 'Road Accidents - Mar.', 'Road Accidents - Apr.', 'Road Accidents - May', 'Road Accidents - Jun.', 'Road Accidents - Jul.', 'Road Accidents - Aug.', 'Road Accidents - Sep.', 'Road Accidents - Oct.', 'Road Accidents - Nov.', 'Road Accidents - Dec.', 'Road Accidents - Total', 'Railway Crossing Accidents - Jan.', 'Railway Crossing Accidents - Feb.', 'Railway Crossing Accidents - Mar.', 'Railway Crossing Accidents - Apr.', 'Railway Crossing Accidents - May', 'Railway Crossing Accidents - Jun.', 'Railway Crossing Accidents - Jul.', 'Railway Crossing Accidents - Aug.', 'Railway Crossing Accidents - Sep.', 'Railway Crossing Accidents - Oct.', 'Railway Crossing Accidents - Nov.', 'Railway Crossing Accidents - Dec.', 'Railway Crossing Accidents - Total', 'Railway Accidents - Jan.', 'Railway Accidents - Feb.', 'Railway Accidents - Mar.', 'Railway Accidents - Apr.', 

In [117]:
# Clean Table_1A.5 - Month of Occurrence (Road Accidents only)
all_years_month = []

# Month column mapping for each year type
MONTH_MAPPING = {
    "January": ["Road Accidents - Jan.", "Road Accidents - Jan", "Road Accidents_Jan", "Road Accidents - January"],
    "February": ["Road Accidents - Feb.", "Road Accidents - Feb", "Road Accidents_Feb", "Road Accidents - February"],
    "March": ["Road Accidents - Mar.", "Road Accidents - Mar", "Road Accidents_Mar", "Road Accidents - March"],
    "April": ["Road Accidents - Apr.", "Road Accidents - April", "Road Accidents_April", "Road Accidents - Apr"],
    "May": ["Road Accidents - May", "Road Accidents_May"],
    "June": ["Road Accidents - Jun.", "Road Accidents - June", "Road Accidents_June"],
    "July": ["Road Accidents - Jul.", "Road Accidents - July", "Road Accidents_July"],
    "August": ["Road Accidents - Aug.", "Road Accidents - Aug", "Road Accidents_Aug", "Road Accidents - August"],
    "September": ["Road Accidents - Sep.", "Road Accidents - Sept", "Road Accidents_Sept", "Road Accidents - September"],
    "October": ["Road Accidents - Oct.", "Road Accidents - Oct", "Road Accidents_Oct", "Road Accidents - October"],
    "November": ["Road Accidents - Nov.", "Road Accidents - Nov", "Road Accidents_Nov", "Road Accidents - November"],
    "December": ["Road Accidents - Dec.", "Road Accidents - Dec", "Road Accidents_Dec", "Road Accidents - December"],
}

for year in YEARS:
    df = all_files[year]["Table_1A.5"]["df"].copy()
    df.columns = df.columns.str.strip()

    # Detect column positions
    cat_col, name_col = detect_transport_cols(df, year)

    # Filter only State and UT rows
    if cat_col is not None:
        df = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    else:
        df = df[df[name_col].astype(str).str.strip().str.upper().isin(VALID_STATE_UT_KEYS)]

    # Remove subtotal rows
    df = df[~df[name_col].astype(str).str.strip().str.upper().str.startswith("TOTAL")]

    # Build cleaned dataframe
    df_clean = df[[name_col]].copy()
    df_clean.columns = ["State/UT"]

    for new_col, source_cols in MONTH_MAPPING.items():
        available = [c for c in source_cols if c in df.columns]
        if available:
            df_clean[new_col] = df[available[0]].apply(pd.to_numeric, errors="coerce")
        else:
            df_clean[new_col] = 0
            print(f"WARNING: {year} — No source column found for {new_col}")

    # Standardize State/UT names
    df_clean["State/UT"] = df_clean["State/UT"].astype(str).str.strip().str.upper().map(STATE_UT_NAMES)

    # Drop unmapped rows
    df_clean = df_clean.dropna(subset=["State/UT"])

    # Add year
    df_clean["Year"] = year

    # Merge Dadra & Daman and J&K & Ladakh if separate
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False).sum()

    all_years_month.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 35 States/UTs processed.
2016 — 35 States/UTs processed.
2017 — 35 States/UTs processed.
2018 — 35 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 35 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 35 States/UTs processed.

All years processed!


In [118]:
# Merge all years and add S.No.
final_month_df = pd.concat(all_years_month, ignore_index=True)

# Sort by Year then alphabetically by State/UT
final_month_df = final_month_df.sort_values(["Year", "State/UT"]).reset_index(drop=True)

# Add continuous S.No.
final_month_df.insert(0, "S.No.", range(1, len(final_month_df) + 1))

# Reorder columns
final_month_df = final_month_df[["S.No.", "State/UT", "Year",
                                  "January", "February", "March", "April",
                                  "May", "June", "July", "August",
                                  "September", "October", "November", "December"]]

# Save to processed folder
final_month_df.to_csv("../data/processed/month_of_occurrence_2015_2022.csv", index=False)

print(f"Total rows: {len(final_month_df)}")
print(f"Years: {final_month_df['Year'].unique()}")
print(f"\nFirst 5 rows:")
print(final_month_df.head())

Total rows: 280
Years: [2015 2016 2017 2018 2019 2020 2021 2022]

First 5 rows:
   S.No.                     State/UT  Year  January  February  March  April  \
0      1  Andaman and Nicobar Islands  2015       27        23     23     17   
1      2               Andhra Pradesh  2015     1868      1970   1917   2041   
2      3            Arunachal Pradesh  2015       26        28     26     15   
3      4                        Assam  2015      764       670    658    551   
4      5                        Bihar  2015      713       753    806    817   

    May  June  July  August  September  October  November  December  
0    25    26    18      15         26       15        23        20  
1  2067  1896  2086    1793       1758     1824      1736      1883  
2    12    14    23      24         24       22        40        30  
3   546   544   511     524        508      609       520       554  
4  1105   911   718     723        683      638       793       907  


In [119]:
# Inspect Table_1A.6 structure for all years
for year in YEARS:
    df = all_files[year]["Table_1A.6"]["df"].copy()
    df.columns = df.columns.str.strip()
    print(f"\n{year} — Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    


2015 — Shape: (93, 39)
Columns: ['Sl. No.', 'Category', 'State/UT/City', 'Road Accidents - 0000 hrs to 0300 hrs. (Night)', 'Road Accidents - 0300 hrs to 0600 hrs. (Night)', 'Road Accidents - 0600 hrs to 0900 hrs (Day)', 'Road Accidents - 0900 hrs to 1200 hrs (Day)', 'Road Accidents - 1200 hrs to 1500 hrs (Day)', 'Road Accidents - 1500 hrs to 1800 hrs (Day)', 'Road Accidents - 1800 hrs to 2100hrs (Night)', 'Road Accidents - 2100 hrs. to 2400hrs (Night)', 'Road Accidents - Total', 'Railway Crossing Accidents - 0000 hrs to 0300 hrs. (Night)', 'Railway Crossing Accidents - 0300 hrs to 0600 hrs. (Night)', 'Railway Crossing Accidents - 0600 hrs to 0900 hrs (Day)', 'Railway Crossing Accidents - 0900 hrs to 1200 hrs (Day)', 'Railway Crossing Accidents - 1200 hrs to 1500 hrs (Day)', 'Railway Crossing Accidents - 1500 hrs to 1800 hrs (Day)', 'Railway Crossing Accidents - 1800 hrs to 2100 hrs (Night)', 'Railway Crossing Accidents - 2100 hrs. to 2400 hrs (Night)', 'Railway Crossing Accidents - To

In [120]:
# Reload 2016 Table_1A.6
file_path_2016 = os.path.join(BASE_PATH, "2016", "ADSI_2016_Table_1A.6.csv")
df_2016 = pd.read_csv(file_path_2016, encoding='utf-8')
all_files[2016]["Table_1A.6"]["df"] = df_2016
print(f"2016 reloaded — Shape: {df_2016.shape}")
print(f"Columns: {list(df_2016.columns)}")

2016 reloaded — Shape: (93, 39)
Columns: ['Sl. No.', 'Category', 'State/UT/City', 'Road Accidents - 0000 hrs to 0300 hrs. (Night)', 'Road Accidents - 0300 hrs to 0600 hrs. (Night)', 'Road Accidents - 0600 hrs to 0900 hrs (Day)', 'Road Accidents - 0900 hrs to 1200 hrs (Day)', 'Road Accidents - 1200 hrs to 1500 hrs (Day)', 'Road Accidents - 1500 hrs to 1800 hrs (Day)', 'Road Accidents - 1800 hrs to 2100hrs (Night)', 'Road Accidents - 2100 hrs. to 2400hrs (Night)', 'Road Accidents - Total', 'Railway Crossing Accidents - 0000 hrs to 0300 hrs. (Night)', 'Railway Crossing Accidents - 0300 hrs to 0600 hrs. (Night)', 'Railway Crossing Accidents - 0600 hrs to 0900 hrs (Day)', 'Railway Crossing Accidents - 0900 hrs to 1200 hrs (Day)', 'Railway Crossing Accidents - 1200 hrs to 1500 hrs (Day)', 'Railway Crossing Accidents - 1500 hrs to 1800 hrs (Day)', 'Railway Crossing Accidents - 1800 hrs to 2100 hrs (Night)', 'Railway Crossing Accidents - 2100 hrs. to 2400 hrs (Night)', 'Railway Crossing Accide

In [121]:
# Clean Table_1A.6 - Time of Occurrence (Road Accidents only)
all_years_time = []

TIME_MAPPING = {
    "0000 hrs to 0300 hrs (Night)": [
        "Road Accidents - 0000 hrs to 0300 hrs. (Night)",
        "Road Accidents_0000 hrs to 0300 hrs. (Night)"
    ],
    "0300 hrs to 0600 hrs (Night)": [
        "Road Accidents - 0300 hrs to 0600 hrs. (Night)",
        "Road Accidents_0300 hrs to 0600 hrs. (Night)"
    ],
    "0600 hrs to 0900 hrs (Day)": [
        "Road Accidents - 0600 hrs to 0900 hrs (Day)",
        "Road Accidents - 0600 hrs to 0900 hrs. (Day)",
        "Road Accidents_0600 hrs to 0900 hrs. (Day)"
    ],
    "0900 hrs to 1200 hrs (Day)": [
        "Road Accidents - 0900 hrs to 1200 hrs (Day)",
        "Road Accidents - 0900 hrs to 1200 hrs. (Day)",
        "Road Accidents_0900 hrs to 1200 hrs. (Day)"
    ],
    "1200 hrs to 1500 hrs (Day)": [
        "Road Accidents - 1200 hrs to 1500 hrs (Day)",
        "Road Accidents - 1200 hrs to 1500 hrs. (Day)",
        "Road Accidents_1200 hrs to 1500 hrs. (Day)"
    ],
    "1500 hrs to 1800 hrs (Day)": [
        "Road Accidents - 1500 hrs to 1800 hrs (Day)",
        "Road Accidents - 1500 hrs to 1800 hrs. (Day)",
        "Road Accidents_1500 hrs to 1800 hrs. (Day)"
    ],
    "1800 hrs to 2100 hrs (Night)": [
        "Road Accidents - 1800 hrs to 2100 hrs (Night)",
        "Road Accidents - 1800 hrs to 2100hrs (Night)",
        "Road Accidents_1800 hrs to 2100 hrs. (Night)"
    ],
    "2100 hrs to 2400 hrs (Night)": [
        "Road Accidents - 2100 hrs to 2400 hrs (Night)",
        "Road Accidents - 2100 hrs. to 2400hrs (Night)",
        "Road Accidents - 2100 hrs. to 2400hrs(Night)",
        "Road Accidents_2100 hrs to 2400 hrs. (Night)"
    ],
}

for year in YEARS:
    df = all_files[year]["Table_1A.6"]["df"].copy()
    df.columns = df.columns.str.strip()

    # Detect column positions
    cat_col, name_col = detect_transport_cols(df, year)

    # Filter only State and UT rows
    if cat_col is not None:
        df = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    else:
        df = df[df[name_col].astype(str).str.strip().str.upper().isin(VALID_STATE_UT_KEYS)]

    # Remove subtotal rows
    df = df[~df[name_col].astype(str).str.strip().str.upper().str.startswith("TOTAL")]

    # Build cleaned dataframe
    df_clean = df[[name_col]].copy()
    df_clean.columns = ["State/UT"]

    for new_col, source_cols in TIME_MAPPING.items():
        available = [c for c in source_cols if c in df.columns]
        if available:
            df_clean[new_col] = df[available[0]].apply(pd.to_numeric, errors="coerce")
        else:
            df_clean[new_col] = 0
            print(f"WARNING: {year} — No source column found for {new_col}")

    # Standardize State/UT names
    df_clean["State/UT"] = df_clean["State/UT"].astype(str).str.strip().str.upper().map(STATE_UT_NAMES)

    # Drop unmapped rows
    df_clean = df_clean.dropna(subset=["State/UT"])

    # Add year
    df_clean["Year"] = year

    # Merge Dadra & Daman and J&K & Ladakh if separate
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False).sum()

    all_years_time.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 35 States/UTs processed.
2016 — 35 States/UTs processed.
2017 — 35 States/UTs processed.
2018 — 35 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 35 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 34 States/UTs processed.

All years processed!


In [122]:
# Debug 2020, 2021 night columns and 2022 missing State/UT
for year in [2020, 2021, 2022]:
    df = all_files[year]["Table_1A.6"]["df"].copy()
    df.columns = df.columns.str.strip()
    night_cols = [c for c in df.columns if "1800" in c or "2100" in c]
    print(f"\n{year} — Night columns: {night_cols}")

# Check 2022 missing
reference = set(all_years_time[0]["State/UT"].unique())
present_2022 = set(all_years_time[7]["State/UT"].unique())
print(f"\n2022 — Missing: {reference - present_2022}")


2020 — Night columns: ['Road Accidents - 1500 hrs to 1800 hrs. (Day)', 'Road Accidents - 1800 hrs to 2100 hrs. (Night)', 'Road Accidents - 2100 hrs to 2400 hrs. (Night)', 'Railway Crossing Accidents - 1500 hrs to 1800 hrs. (Day)', 'Railway Crossing Accidents - 1800 hrs to 2100 hrs. (Night)', 'Railway Crossing Accidents - 2100 hrs to 2400 hrs. (Night)', 'Railway Accidents - 1500 hrs to 1800 hrs. (Day)', 'Railway Accidents - 1800 hrs to 2100 hrs. (Night)', 'Railway Accidents - 2100 hrs to 2400 hrs. (Night)', 'Total (Traffic Accidents) - 1500 hrs to 1800 hrs. (Day)', 'Total (Traffic Accidents) - 1800 hrs to 2100 hrs. (Night)', 'Total (Traffic Accidents) - 2100 hrs to 2400 hrs. (Night)']

2021 — Night columns: ['Road Accidents - 1500 hrs to 1800 hrs. (Day)', 'Road Accidents - 1800 hrs to 2100 hrs. (Night)', 'Road Accidents - 2100 hrs to 2400 hrs. (Night)', 'Railway Crossing Accidents - 1500 hrs to 1800 hrs. (Day)', 'Railway Crossing Accidents - 1800 hrs to 2100 hrs. (Night)', 'Railway Cro

In [123]:
# Updated TIME_MAPPING with fixes
TIME_MAPPING = {
    "0000 hrs to 0300 hrs (Night)": [
        "Road Accidents - 0000 hrs to 0300 hrs. (Night)",
        "Road Accidents_0000 hrs to 0300 hrs. (Night)"
    ],
    "0300 hrs to 0600 hrs (Night)": [
        "Road Accidents - 0300 hrs to 0600 hrs. (Night)",
        "Road Accidents_0300 hrs to 0600 hrs. (Night)"
    ],
    "0600 hrs to 0900 hrs (Day)": [
        "Road Accidents - 0600 hrs to 0900 hrs (Day)",
        "Road Accidents - 0600 hrs to 0900 hrs. (Day)",
        "Road Accidents_0600 hrs to 0900 hrs. (Day)"
    ],
    "0900 hrs to 1200 hrs (Day)": [
        "Road Accidents - 0900 hrs to 1200 hrs (Day)",
        "Road Accidents - 0900 hrs to 1200 hrs. (Day)",
        "Road Accidents_0900 hrs to 1200 hrs. (Day)"
    ],
    "1200 hrs to 1500 hrs (Day)": [
        "Road Accidents - 1200 hrs to 1500 hrs (Day)",
        "Road Accidents - 1200 hrs to 1500 hrs. (Day)",
        "Road Accidents_1200 hrs to 1500 hrs. (Day)"
    ],
    "1500 hrs to 1800 hrs (Day)": [
        "Road Accidents - 1500 hrs to 1800 hrs (Day)",
        "Road Accidents - 1500 hrs to 1800 hrs. (Day)",
        "Road Accidents_1500 hrs to 1800 hrs. (Day)"
    ],
    "1800 hrs to 2100 hrs (Night)": [
        "Road Accidents - 1800 hrs to 2100 hrs (Night)",
        "Road Accidents - 1800 hrs to 2100 hrs. (Night)",
        "Road Accidents - 1800 hrs to 2100hrs (Night)",
        "Road Accidents_1800 hrs to 2100 hrs. (Night)"
    ],
    "2100 hrs to 2400 hrs (Night)": [
        "Road Accidents - 2100 hrs to 2400 hrs (Night)",
        "Road Accidents - 2100 hrs to 2400 hrs. (Night)",
        "Road Accidents - 2100 hrs. to 2400hrs (Night)",
        "Road Accidents - 2100 hrs. to 2400hrs(Night)",
        "Road Accidents_2100 hrs to 2400 hrs. (Night)"
    ],
}

# Fix 2022 missing — check raw name
df_2022 = all_files[2022]["Table_1A.6"]["df"].copy()
df_2022.columns = df_2022.columns.str.strip()
name_col_2022 = df_2022.columns[1]
names = df_2022[name_col_2022].astype(str).str.strip().tolist()
dadra = [n for n in names if "DADRA" in n.upper() or "DAMAN" in n.upper() or "HAVELI" in n.upper()]
print(f"2022 Dadra raw name: {dadra}")

2022 Dadra raw name: ['Andaman and Nicobar Islands', 'Dadra and Nicobar Haveli and Daman and Diu']


In [124]:
# Add typo fix to STATE_UT_NAMES
STATE_UT_NAMES["DADRA AND NICOBAR HAVELI AND DAMAN AND DIU"] = "Dadra and Nagar Haveli and Daman and Diu"
print("Mapping updated!")


Mapping updated!


In [125]:
# Clean Table_1A.6 - Time of Occurrence (Road Accidents only) - FIXED
all_years_time = []

TIME_MAPPING_FIXED = {
    "0000 hrs to 0300 hrs (Night)": [
        "Road Accidents - 0000 hrs to 0300 hrs. (Night)",
        "Road Accidents_0000 hrs to 0300 hrs. (Night)"
    ],
    "0300 hrs to 0600 hrs (Night)": [
        "Road Accidents - 0300 hrs to 0600 hrs. (Night)",
        "Road Accidents_0300 hrs to 0600 hrs. (Night)"
    ],
    "0600 hrs to 0900 hrs (Day)": [
        "Road Accidents - 0600 hrs to 0900 hrs (Day)",
        "Road Accidents - 0600 hrs to 0900 hrs. (Day)",
        "Road Accidents_0600 hrs to 0900 hrs. (Day)"
    ],
    "0900 hrs to 1200 hrs (Day)": [
        "Road Accidents - 0900 hrs to 1200 hrs (Day)",
        "Road Accidents - 0900 hrs to 1200 hrs. (Day)",
        "Road Accidents_0900 hrs to 1200 hrs. (Day)"
    ],
    "1200 hrs to 1500 hrs (Day)": [
        "Road Accidents - 1200 hrs to 1500 hrs (Day)",
        "Road Accidents - 1200 hrs to 1500 hrs. (Day)",
        "Road Accidents_1200 hrs to 1500 hrs. (Day)"
    ],
    "1500 hrs to 1800 hrs (Day)": [
        "Road Accidents - 1500 hrs to 1800 hrs (Day)",
        "Road Accidents - 1500 hrs to 1800 hrs. (Day)",
        "Road Accidents_1500 hrs to 1800 hrs. (Day)"
    ],
    "1800 hrs to 2100 hrs (Night)": [
        "Road Accidents - 1800 hrs to 2100 hrs (Night)",
        "Road Accidents - 1800 hrs to 2100 hrs. (Night)",
        "Road Accidents - 1800 hrs to 2100hrs (Night)",
        "Road Accidents_1800 hrs to 2100 hrs. (Night)"
    ],
    "2100 hrs to 2400 hrs (Night)": [
        "Road Accidents - 2100 hrs to 2400 hrs (Night)",
        "Road Accidents - 2100 hrs to 2400 hrs. (Night)",
        "Road Accidents - 2100 hrs. to 2400hrs (Night)",
        "Road Accidents - 2100 hrs. to 2400hrs(Night)",
        "Road Accidents_2100 hrs to 2400 hrs. (Night)"
    ],
}

for year in YEARS:
    df = all_files[year]["Table_1A.6"]["df"].copy()
    df.columns = df.columns.str.strip()

    cat_col, name_col = detect_transport_cols(df, year)

    if cat_col is not None:
        df = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    else:
        df = df[df[name_col].astype(str).str.strip().str.upper().isin(VALID_STATE_UT_KEYS)]

    df = df[~df[name_col].astype(str).str.strip().str.upper().str.startswith("TOTAL")]

    df_clean = df[[name_col]].copy()
    df_clean.columns = ["State/UT"]

    for new_col, source_cols in TIME_MAPPING_FIXED.items():
        available = [c for c in source_cols if c in df.columns]
        if available:
            df_clean[new_col] = df[available[0]].apply(pd.to_numeric, errors="coerce")
        else:
            df_clean[new_col] = 0
            print(f"WARNING: {year} — No source column found for {new_col}")

    df_clean["State/UT"] = df_clean["State/UT"].astype(str).str.strip().str.upper().map(STATE_UT_NAMES)
    df_clean = df_clean.dropna(subset=["State/UT"])
    df_clean["Year"] = year
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False).sum()

    all_years_time.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 35 States/UTs processed.
2016 — 35 States/UTs processed.
2017 — 35 States/UTs processed.
2018 — 35 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 35 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 34 States/UTs processed.

All years processed!


In [126]:
# Debug 2022 missing
reference = set(all_years_time[0]["State/UT"].unique())
present_2022 = set(all_years_time[7]["State/UT"].unique())
print(f"2022 — Missing: {reference - present_2022}")

# Check raw name in 2022
df_2022 = all_files[2022]["Table_1A.6"]["df"].copy()
df_2022.columns = df_2022.columns.str.strip()
name_col_2022 = df_2022.columns[1]
print(f"\n2022 name_col: {name_col_2022}")
print(f"All raw names: {df_2022[name_col_2022].tolist()}")

2022 — Missing: {'Dadra and Nagar Haveli and Daman and Diu'}

2022 name_col: State/UT
All raw names: ['Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chhattisgarh', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal', 'Total (States)', 'Andaman and Nicobar Islands', 'Chandigarh', 'Dadra and Nicobar Haveli and Daman and Diu', 'Delhi (UT)', 'Jammu and Kashmir', 'Ladakh', 'Lakshadweep', 'Puducherry', 'Total (UTs)', 'Total (All India)', 'Agra', 'Ahmedabad', 'Amritsar', 'Asansol', 'Aurangabad', 'Bengaluru', 'Bhopal', 'Chandigarh (City)', 'Chennai', 'Coimbatore', 'Delhi (City)', 'Dhanbad', 'Durg Bhilainagar', 'Faridabad', 'Ghaziabad', 'Gwalior', 'Hyderabad', 'Indore', 'Jabalpur', 'Jaipur', 'Jamshedpur', 'Jodhpur', 'Kannur', 'Kanpur', 'Kochi', 'Ko

In [127]:
# Update VALID_STATE_UT_KEYS
VALID_STATE_UT_KEYS = set(STATE_UT_NAMES.keys())
print("VALID_STATE_UT_KEYS updated!")


VALID_STATE_UT_KEYS updated!


In [128]:
# Merge all years and add S.No.
final_time_df = pd.concat(all_years_time, ignore_index=True)

# Sort by Year then alphabetically by State/UT
final_time_df = final_time_df.sort_values(["Year", "State/UT"]).reset_index(drop=True)

# Add continuous S.No.
final_time_df.insert(0, "S.No.", range(1, len(final_time_df) + 1))

# Reorder columns
final_time_df = final_time_df[["S.No.", "State/UT", "Year",
                                "0000 hrs to 0300 hrs (Night)",
                                "0300 hrs to 0600 hrs (Night)",
                                "0600 hrs to 0900 hrs (Day)",
                                "0900 hrs to 1200 hrs (Day)",
                                "1200 hrs to 1500 hrs (Day)",
                                "1500 hrs to 1800 hrs (Day)",
                                "1800 hrs to 2100 hrs (Night)",
                                "2100 hrs to 2400 hrs (Night)"]]

# Save to processed folder
final_time_df.to_csv("../data/processed/time_of_occurrence_2015_2022.csv", index=False)

print(f"Total rows: {len(final_time_df)}")
print(f"Years: {final_time_df['Year'].unique()}")
print(f"\nFirst 5 rows:")
print(final_time_df.head())

Total rows: 279
Years: [2015 2016 2017 2018 2019 2020 2021 2022]

First 5 rows:
   S.No.                     State/UT  Year  0000 hrs to 0300 hrs (Night)  \
0      1  Andaman and Nicobar Islands  2015                             1   
1      2               Andhra Pradesh  2015                          2160   
2      3            Arunachal Pradesh  2015                             7   
3      4                        Assam  2015                           525   
4      5                        Bihar  2015                           719   

   0300 hrs to 0600 hrs (Night)  0600 hrs to 0900 hrs (Day)  \
0                             3                          25   
1                          2080                        2492   
2                             9                          23   
3                           485                         931   
4                           992                        1512   

   0900 hrs to 1200 hrs (Day)  1200 hrs to 1500 hrs (Day)  \
0                

In [129]:
# Inspect Table_1A.7 structure for all years
for year in YEARS:
    df = all_files[year]["Table_1A.7"]["df"].copy()
    df.columns = df.columns.str.strip()
    print(f"\n{year} — Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")


2015 — Shape: (93, 18)
Columns: ['Sl. No.', 'Category', 'State/UT/City', 'National Highways - Cases', 'National Highways - Injured', 'National Highways - Died', 'State Highways - Cases', 'State Highways - Injured', 'State Highways - Died', 'Expressways - Cases', 'Expressways - Injured', 'Expressways - Died', 'Other Roads - Cases', 'Other Roads - Injured', 'Other Roads - Died', 'Total - Cases', 'Total - Injured', 'Total - Died']

2016 — Shape: (93, 18)
Columns: ['Sl. No.', 'Category', 'State/UT/City', 'National Highways - Cases', 'National Highways - Injured', 'National Highways - Died', 'State Highways - Cases', 'State Highways - Injured', 'State Highways - Died', 'Expressways - Cases', 'Expressways - Injured', 'Expressways - Died', 'Other Roads - Cases', 'Other Roads - Injured', 'Other Roads - Died', 'Total - Cases', 'Total - Injured', 'Total - Died']

2017 — Shape: (93, 18)
Columns: ['Sl. No.', 'Category', 'State/UT/City', 'National Highways - Cases', 'National Highways - Injured', 

In [130]:
# Clean Table_1A.7 - Road Classification (Died only)
all_years_road = []

ROAD_MAPPING = {
    "National Highways - Died": [
        "National Highways - Died",
        "National Highways_Died"
    ],
    "State Highways - Died": [
        "State Highways - Died",
        "State Highways_Died"
    ],
    "Expressways - Died": [
        "Expressways - Died",
        "Expressways_Died"
    ],
    "Other Roads - Died": [
        "Other Roads - Died",
        "Other Roads_Died"
    ],
    "Total - Died": [
        "Total - Died",
        "Total_Died"
    ],
}

for year in YEARS:
    df = all_files[year]["Table_1A.7"]["df"].copy()
    df.columns = df.columns.str.strip()

    cat_col, name_col = detect_transport_cols(df, year)

    if cat_col is not None:
        df = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    else:
        df = df[df[name_col].astype(str).str.strip().str.upper().isin(VALID_STATE_UT_KEYS)]

    df = df[~df[name_col].astype(str).str.strip().str.upper().str.startswith("TOTAL")]

    df_clean = df[[name_col]].copy()
    df_clean.columns = ["State/UT"]

    for new_col, source_cols in ROAD_MAPPING.items():
        available = [c for c in source_cols if c in df.columns]
        if available:
            df_clean[new_col] = df[available[0]].apply(pd.to_numeric, errors="coerce")
        else:
            df_clean[new_col] = 0
            print(f"WARNING: {year} — No source column found for {new_col}")

    df_clean["State/UT"] = df_clean["State/UT"].astype(str).str.strip().str.upper().map(STATE_UT_NAMES)
    df_clean = df_clean.dropna(subset=["State/UT"])
    df_clean["Year"] = year
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False).sum()

    all_years_road.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 35 States/UTs processed.
2016 — 35 States/UTs processed.
2017 — 35 States/UTs processed.
2018 — 35 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 35 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 35 States/UTs processed.

All years processed!


In [131]:
# Merge all years and add S.No.
final_road_df = pd.concat(all_years_road, ignore_index=True)

# Sort by Year then alphabetically by State/UT
final_road_df = final_road_df.sort_values(["Year", "State/UT"]).reset_index(drop=True)

# Add continuous S.No.
final_road_df.insert(0, "S.No.", range(1, len(final_road_df) + 1))

# Reorder columns
final_road_df = final_road_df[["S.No.", "State/UT", "Year",
                                "National Highways - Died",
                                "State Highways - Died",
                                "Expressways - Died",
                                "Other Roads - Died",
                                "Total - Died"]]

# Save to processed folder
final_road_df.to_csv("../data/processed/road_classification_2015_2022.csv", index=False)

print(f"Total rows: {len(final_road_df)}")
print(f"Years: {final_road_df['Year'].unique()}")
print(f"\nFirst 5 rows:")
print(final_road_df.head())

Total rows: 280
Years: [2015 2016 2017 2018 2019 2020 2021 2022]

First 5 rows:
   S.No.                     State/UT  Year  National Highways - Died  \
0      1  Andaman and Nicobar Islands  2015                         4   
1      2               Andhra Pradesh  2015                      3062   
2      3            Arunachal Pradesh  2015                        52   
3      4                        Assam  2015                      1312   
4      5                        Bihar  2015                      2166   

   State Highways - Died  Expressways - Died  Other Roads - Died  Total - Died  
0                      5                   0                  14            23  
1                   2025                   0                3210          8297  
2                     82                   0                  43           177  
3                    570                   0                 502          2384  
4                   1906                 111                1317          55

In [132]:
# Inspect Table_1A.11 structure for all years
for year in YEARS:
    df = all_files[year]["Table_1A.11"]["df"].copy()
    df.columns = df.columns.str.strip()
    print(f"\n{year} — Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")


2015 — Shape: (93, 67)
Columns: ['Sl. No.', 'Category', 'State/UT/City', 'Rural Area (Near School/College/Educational Institution) - Male', 'Rural Area (Near School/College/Educational Institution) - Female', 'Rural Area (Near School/College/Educational Institution) - Transgender', 'Rural Area (Near School/College/Educational Institution) - Total', 'Rural Area (Near Residential Area) - Male', 'Rural Area (Near Residential Area) - Female', 'Rural Area (Near Residential Area) - Transgender', 'Rural Area (Near Residential Area) - Total', 'Rural Area (Near Religious Place) - Male', 'Rural Area (Near Religious Place) - Female', 'Rural Area (Near Religious Place) - Transgender', 'Rural Area (Near Religious Place) - Total', 'Rural Area (Near Recreation Place/Cinema Hall) - Male', 'Rural Area (Near Recreation Place/Cinema Hall) - Female', 'Rural Area (Near Recreation Place/Cinema Hall) - Transgender', 'Rural Area (Near Recreation Place/Cinema Hall) - Total', 'Rural Area (Near Factory) - Male'

In [133]:
# Reload 2016, 2021, 2022 Table_1A.11
for year in [2016, 2021, 2022]:
    file_path = os.path.join(BASE_PATH, str(year), f"ADSI_{year}_Table_1A.11.csv")
    df = pd.read_csv(file_path, encoding='utf-8')
    all_files[year]["Table_1A.11"]["df"] = df
    print(f"{year} reloaded — Shape: {df.shape}")
    print(f"Columns: {list(df.columns[:5])}\n")

2016 reloaded — Shape: (93, 67)
Columns: ['Sl. No.', 'Category', 'State/UT/City', 'Rural Area (Near School/College/Educational Institution) - Male', 'Rural Area (Near School/College/Educational Institution) - Female']

2021 reloaded — Shape: (93, 66)
Columns: ['Category', 'State/UT/City', 'Rural Area(Near School/College/Educational Institution) - Male', 'Rural Area(Near School/College/Educational Institution) - Female', 'Rural Area(Near School/College/Educational Institution) - Transgender']

2022 reloaded — Shape: (93, 66)
Columns: ['Sl. No.', 'State/UT/Cities', 'Rural Area (Near School/College/Educational Institution) - Male', 'Rural Area (Near School/College/Educational Institution) - Female', 'Rural Area (Near School/College/Educational Institution) - Transgender']



In [134]:
# Clean Table_1A.11 - Place of Occurrence (Total only)
all_years_place = []

PLACE_MAPPING = {
    "Rural Area - School/College - Total": [
        "Rural Area (Near School/College/Educational Institution) - Total",
        "Rural Area(Near School/College/Educational Institution) - Total",
        "Rural Area(Near School/College/Educational Institution)_Total"
    ],
    "Rural Area - Residential - Total": [
        "Rural Area (Near Residential Area) - Total",
        "Rural Area (Near Residential Area)_Total"
    ],
    "Rural Area - Religious - Total": [
        "Rural Area (Near Religious Place) - Total",
        "Rural Area (Near Religious Place)_Total"
    ],
    "Rural Area - Recreation - Total": [
        "Rural Area (Near Recreation Place/Cinema Hall) - Total",
        "Rural Area (Near Recreation Place/ Cinema Hall) - Total",
        "Rural Area (Near Recreation Place/ Cinema Hall)_Total"
    ],
    "Rural Area - Factory - Total": [
        "Rural Area (Near Factory) - Total",
        "Rural Area (Near Factory)_Total"
    ],
    "Rural Area - Others - Total": [
        "Rural Area (Others) - Total",
        "Rural Area (Others)_Total"
    ],
    "Rural Area - Sub Total": [
        "Rural Area (Sub Total) - Total",
        "Rural Area (Sub Total)_Total"
    ],
    "Urban Area - School/College - Total": [
        "Urban Area (Near School/College/Educational Institution) - Total",
        "Urban Area (Near School/College/ Educational Institution) - Total",
        "Urban Area (Near School/College/ Educational Institution)_Total"
    ],
    "Urban Area - Residential - Total": [
        "Urban Area (Near Residential Area) - Total",
        "Urban Area (Near Residential Area)_Total"
    ],
    "Urban Area - Religious - Total": [
        "Urban Area (Near Religious Place) - Total",
        "Urban Area (Near Religious Place)_Total"
    ],
    "Urban Area - Recreation - Total": [
        "Urban Area (Near Recreation Place/Cinema Hall) - Total",
        "Urban Area (Near Recreation Place/ Cinema Hall) - Total",
        "Urban Area (Near Recreation Place/ Cinema Hall)_Total"
    ],
    "Urban Area - Factory - Total": [
        "Urban Area (Near Factory/Industrial Area) - Total",
        "Urban Area (Near Factory/Industrial Area)_Total"
    ],
    "Urban Area - Pedestrian Crossing - Total": [
        "Urban Area (At Pedestrian Crossing) - Total",
        "Urban Area (At Pedestrian Crossing)_Total"
    ],
    "Urban Area - Others - Total": [
        "Urban Area (Others) - Total",
        "Urban Area (Others)_Total"
    ],
    "Urban Area - Sub Total": [
        "Urban Area (Sub Total) - Total",
        "Urban Area (Sub Total)_Total"
    ],
    "Grand Total": [
        "Grand Total",
        "Grand Total - Total",
        "Grand Total_Total"
    ],
}

for year in YEARS:
    df = all_files[year]["Table_1A.11"]["df"].copy()
    df.columns = df.columns.str.strip()

    cat_col, name_col = detect_transport_cols(df, year)

    if cat_col is not None:
        df = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    else:
        df = df[df[name_col].astype(str).str.strip().str.upper().isin(VALID_STATE_UT_KEYS)]

    df = df[~df[name_col].astype(str).str.strip().str.upper().str.startswith("TOTAL")]

    df_clean = df[[name_col]].copy()
    df_clean.columns = ["State/UT"]

    for new_col, source_cols in PLACE_MAPPING.items():
        available = [c for c in source_cols if c in df.columns]
        if available:
            df_clean[new_col] = df[available[0]].apply(pd.to_numeric, errors="coerce")
        else:
            df_clean[new_col] = 0
            print(f"WARNING: {year} — No source column found for {new_col}")

    df_clean["State/UT"] = df_clean["State/UT"].astype(str).str.strip().str.upper().map(STATE_UT_NAMES)
    df_clean = df_clean.dropna(subset=["State/UT"])
    df_clean["Year"] = year
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False).sum()

    all_years_place.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 35 States/UTs processed.
2016 — 35 States/UTs processed.
2017 — 35 States/UTs processed.
2018 — 35 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 35 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 35 States/UTs processed.

All years processed!


In [135]:
# Debug 2022 Rural Residential column
df_2022 = all_files[2022]["Table_1A.11"]["df"].copy()
df_2022.columns = df_2022.columns.str.strip()
rural_cols = [c for c in df_2022.columns if "Rural" in c and "Total" in c]
print(f"2022 Rural Total columns: {rural_cols}")

2022 Rural Total columns: ['Rural Area (Near School/College/Educational Institution) - Total', 'Rural Area(Near Residential Area) - Total', 'Rural Area (Near Religious Place) - Total', 'Rural Area (Near Recreation Place/ Cinema Hall) - Total', 'Rural Area (Near Factory) - Total', 'Rural Area (Others) - Total', 'Rural Area (Sub Total) - Male', 'Rural Area (Sub Total) - Female', 'Rural Area (Sub Total) - Transgender', 'Rural Area (Sub Total) - Total']


In [136]:
# Fix 2022 Rural Residential mapping
PLACE_MAPPING["Rural Area - Residential - Total"].append(
    "Rural Area(Near Residential Area) - Total"
)
print("Mapping updated!")


Mapping updated!


In [137]:
# Clean Table_1A.11 - Place of Occurrence (Total only) - FIXED
all_years_place = []

PLACE_MAPPING_FIXED = {
    "Rural Area - School/College - Total": [
        "Rural Area (Near School/College/Educational Institution) - Total",
        "Rural Area(Near School/College/Educational Institution) - Total",
        "Rural Area(Near School/College/Educational Institution)_Total"
    ],
    "Rural Area - Residential - Total": [
        "Rural Area (Near Residential Area) - Total",
        "Rural Area(Near Residential Area) - Total",
        "Rural Area (Near Residential Area)_Total"
    ],
    "Rural Area - Religious - Total": [
        "Rural Area (Near Religious Place) - Total",
        "Rural Area (Near Religious Place)_Total"
    ],
    "Rural Area - Recreation - Total": [
        "Rural Area (Near Recreation Place/Cinema Hall) - Total",
        "Rural Area (Near Recreation Place/ Cinema Hall) - Total",
        "Rural Area (Near Recreation Place/ Cinema Hall)_Total"
    ],
    "Rural Area - Factory - Total": [
        "Rural Area (Near Factory) - Total",
        "Rural Area (Near Factory)_Total"
    ],
    "Rural Area - Others - Total": [
        "Rural Area (Others) - Total",
        "Rural Area (Others)_Total"
    ],
    "Rural Area - Sub Total": [
        "Rural Area (Sub Total) - Total",
        "Rural Area (Sub Total)_Total"
    ],
    "Urban Area - School/College - Total": [
        "Urban Area (Near School/College/Educational Institution) - Total",
        "Urban Area (Near School/College/ Educational Institution) - Total",
        "Urban Area (Near School/College/ Educational Institution)_Total"
    ],
    "Urban Area - Residential - Total": [
        "Urban Area (Near Residential Area) - Total",
        "Urban Area (Near Residential Area)_Total"
    ],
    "Urban Area - Religious - Total": [
        "Urban Area (Near Religious Place) - Total",
        "Urban Area (Near Religious Place)_Total"
    ],
    "Urban Area - Recreation - Total": [
        "Urban Area (Near Recreation Place/Cinema Hall) - Total",
        "Urban Area (Near Recreation Place/ Cinema Hall) - Total",
        "Urban Area (Near Recreation Place/ Cinema Hall)_Total"
    ],
    "Urban Area - Factory - Total": [
        "Urban Area (Near Factory/Industrial Area) - Total",
        "Urban Area (Near Factory/Industrial Area)_Total"
    ],
    "Urban Area - Pedestrian Crossing - Total": [
        "Urban Area (At Pedestrian Crossing) - Total",
        "Urban Area (At Pedestrian Crossing)_Total"
    ],
    "Urban Area - Others - Total": [
        "Urban Area (Others) - Total",
        "Urban Area (Others)_Total"
    ],
    "Urban Area - Sub Total": [
        "Urban Area (Sub Total) - Total",
        "Urban Area (Sub Total)_Total"
    ],
    "Grand Total": [
        "Grand Total",
        "Grand Total - Total",
        "Grand Total_Total"
    ],
}

for year in YEARS:
    df = all_files[year]["Table_1A.11"]["df"].copy()
    df.columns = df.columns.str.strip()

    cat_col, name_col = detect_transport_cols(df, year)

    if cat_col is not None:
        df = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    else:
        df = df[df[name_col].astype(str).str.strip().str.upper().isin(VALID_STATE_UT_KEYS)]

    df = df[~df[name_col].astype(str).str.strip().str.upper().str.startswith("TOTAL")]

    df_clean = df[[name_col]].copy()
    df_clean.columns = ["State/UT"]

    for new_col, source_cols in PLACE_MAPPING_FIXED.items():
        available = [c for c in source_cols if c in df.columns]
        if available:
            df_clean[new_col] = df[available[0]].apply(pd.to_numeric, errors="coerce")
        else:
            df_clean[new_col] = 0
            print(f"WARNING: {year} — No source column found for {new_col}")

    df_clean["State/UT"] = df_clean["State/UT"].astype(str).str.strip().str.upper().map(STATE_UT_NAMES)
    df_clean = df_clean.dropna(subset=["State/UT"])
    df_clean["Year"] = year
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False).sum()

    all_years_place.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 35 States/UTs processed.
2016 — 35 States/UTs processed.
2017 — 35 States/UTs processed.
2018 — 35 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 35 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 35 States/UTs processed.

All years processed!


In [138]:
# Merge all years and add S.No.
final_place_df = pd.concat(all_years_place, ignore_index=True)

# Sort by Year then alphabetically by State/UT
final_place_df = final_place_df.sort_values(["Year", "State/UT"]).reset_index(drop=True)

# Add continuous S.No.
final_place_df.insert(0, "S.No.", range(1, len(final_place_df) + 1))

# Reorder columns
final_place_df = final_place_df[["S.No.", "State/UT", "Year",
                                  "Rural Area - School/College - Total",
                                  "Rural Area - Residential - Total",
                                  "Rural Area - Religious - Total",
                                  "Rural Area - Recreation - Total",
                                  "Rural Area - Factory - Total",
                                  "Rural Area - Others - Total",
                                  "Rural Area - Sub Total",
                                  "Urban Area - School/College - Total",
                                  "Urban Area - Residential - Total",
                                  "Urban Area - Religious - Total",
                                  "Urban Area - Recreation - Total",
                                  "Urban Area - Factory - Total",
                                  "Urban Area - Pedestrian Crossing - Total",
                                  "Urban Area - Others - Total",
                                  "Urban Area - Sub Total",
                                  "Grand Total"]]

# Save to processed folder
final_place_df.to_csv("../data/processed/place_of_occurrence_2015_2022.csv", index=False)

print(f"Total rows: {len(final_place_df)}")
print(f"Years: {final_place_df['Year'].unique()}")
print(f"\nFirst 5 rows:")
print(final_place_df.head())

Total rows: 280
Years: [2015 2016 2017 2018 2019 2020 2021 2022]

First 5 rows:
   S.No.                     State/UT  Year  \
0      1  Andaman and Nicobar Islands  2015   
1      2               Andhra Pradesh  2015   
2      3            Arunachal Pradesh  2015   
3      4                        Assam  2015   
4      5                        Bihar  2015   

   Rural Area - School/College - Total  Rural Area - Residential - Total  \
0                                    0                                 9   
1                                  289                              1406   
2                                    2                                27   
3                                  225                               481   
4                                  291                               615   

   Rural Area - Religious - Total  Rural Area - Recreation - Total  \
0                               0                                1   
1                             254       

In [139]:
# Debug missing entries
for year, state in [(2022, "ANDAMAN"), (2020, "DADRA")]:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    names = df.iloc[:, 1].astype(str).str.strip().tolist() if year == 2020 else df.iloc[:, 0].astype(str).str.strip().tolist()
    matched = [n for n in names if state in n.upper()]
    print(f"{year} — {state} raw names: {matched}")

2022 — ANDAMAN raw names: []
2020 — DADRA raw names: []


In [140]:
# Debug — check 2022 and 2020 Table_1.2 raw data directly
for year in [2020, 2022]:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    cat_col, name_col = detect_cols(df)
    print(f"\n{year} — name_col: {name_col}")
    names = df[name_col].astype(str).str.strip().tolist()
    matched = [n for n in names if "ANDAMAN" in n.upper() or "DADRA" in n.upper() or "DAMAN" in n.upper()]
    print(f"Matched names: {matched}")


2020 — name_col: State/UT/City (Col. 2)
Matched names: ['D & N Haveli and Daman & Diu']

2022 — name_col: State/UT
Matched names: ['Andamand and Nicobar Islands', 'Dadra and Nagar Haveli and Daman and Diu']


In [141]:
# Fix — add missing entries directly to final processed file
import pandas as pd
import os

# Load the processed file
deaths_path = "../data/processed/total_accidental_deaths_2015_2022.csv"
deaths_df_fix = pd.read_csv(deaths_path)

# Check what values are in STATE_UT_NAMES for these
print(STATE_UT_NAMES.get("D & N HAVELI AND DAMAN & DIU"))
print(STATE_UT_NAMES.get("ANDAMAND AND NICOBAR ISLANDS"))

# Check all_years_data for 2020 and 2022
for i, year in enumerate(YEARS):
    if year in [2020, 2022]:
        df = all_years_data[i]
        matched = df[df["State/UT"].str.contains("Andaman|Dadra", case=False, na=False)]
        print(f"\n{year} — matched rows:")
        print(matched)

Dadra and Nagar Haveli and Daman and Diu
Andaman and Nicobar Islands

2020 — matched rows:
                      State/UT  Year  Total Accidental Deaths
0  Andaman and Nicobar Islands  2020                      0.1

2022 — matched rows:
                                   State/UT  Year  Total Accidental Deaths
6  Dadra and Nagar Haveli and Daman and Diu  2022                        0


In [142]:
# Check 2020 Andaman raw value
year = 2020
df = all_files[year]["Table_1.2"]["df"].copy()
df.columns = df.columns.str.strip()
cat_col, name_col = detect_cols(df)
total_col = find_total_col(df, year)
print(f"total_col: {total_col}")
andaman_row = df[df[name_col].astype(str).str.contains("Andaman|A & N", case=False, na=False)]
print(f"\n2020 Andaman raw row:")
print(andaman_row[[name_col, total_col]])

# Check 2022 Andaman
year = 2022
df = all_files[year]["Table_1.2"]["df"].copy()
df.columns = df.columns.str.strip()
cat_col, name_col = detect_cols(df)
total_col = find_total_col(df, year)
print(f"\ntotal_col: {total_col}")
andaman_row = df[df[name_col].astype(str).str.contains("Andaman", case=False, na=False)]
print(f"\n2022 Andaman raw row:")
print(andaman_row[[name_col, total_col]])

total_col: Percentage Share in Total Deaths (Col. 6)

2020 Andaman raw row:
   State/UT/City (Col. 2)  Percentage Share in Total Deaths (Col. 6)
29          A & N Islands                                        0.1

total_col: Total Number of Accidental Deaths - Forces of Nature - Col. (3)

2022 Andaman raw row:
                        State/UT  \
29  Andamand and Nicobar Islands   

    Total Number of Accidental Deaths - Forces of Nature - Col. (3)  
29                                                  0                


In [143]:
# Check all columns for 2020 and 2022
for year in [2020, 2022]:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    print(f"\n{year} — All columns:")
    for col in df.columns:
        print(f"  {col}")


2020 — All columns:
  Si. No. (Col. 1)
  Category
  State/UT/City (Col. 2)
  Total Number of Accidental Deaths - Forces of Nature (Col. 3)
  Total Number of Accidental Deaths - Other Causes (Col. 4)
  Total Number of Accidental Deaths - Total (Col. 5)
  Percentage Share in Total Deaths (Col. 6)
  Projected Mid-Year Population (in Lakh) (Col. 7)
  Rate of Accidental Deaths (Col. 8) = (Col.5/ Col.7)

2022 — All columns:
  Sl. No.
  Category
  State/UT
  Total Number of Accidental Deaths - Forces of Nature - Col. (3)
  Total Number of Accidental Deaths - Other Causes - Col. (4)
  Total Number of Accidental Deaths - Total - Col. (5)
  Percentage Share in Total Deaths - Col. (6)
  Projected Mid-Year Population* (in Lakh) - Col. (7)
  Rate of Accidental Deaths (Col.5/Col.7)


In [144]:
# Fix find_total_col function
def find_total_col(df, year):
    # 2017 specific
    for col in df.columns:
        if f"Total Accidental Deaths - {year}" in col:
            return col
    # Total Col.5 priority — most years
    for col in df.columns:
        if "Total" in col and ("Col. 5" in col or "Col.5" in col or "Col. (5)" in col):
            return col
    # Total Col.6 — only if Col.5 not found
    for col in df.columns:
        if "Total" in col and ("Col. 6" in col or "Col.6" in col or "Col. (6)" in col):
            return col
    for col in df.columns:
        if "Total" in col and "Number" in col:
            return col
    for col in df.columns:
        if "Total" in col and "Death" in col:
            return col
    return None

print("find_total_col updated!")

find_total_col updated!


In [145]:
# Clean Table_1.2 - Total Accidental Deaths - FINAL FIX
all_years_data = []

VALID_CATEGORIES = ["STATE", "UT", "UNION TERRITORY", "UNION TERRITORIES"]

def detect_cols(df):
    if df.iloc[:, 1].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES).any():
        return df.columns[1], df.columns[2]
    elif df.iloc[:, 0].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES).any():
        return df.columns[0], df.columns[1]
    else:
        return df.columns[1], df.columns[2]

def find_total_col(df, year):
    for col in df.columns:
        if f"Total Accidental Deaths - {year}" in col:
            return col
    for col in df.columns:
        if "Total" in col and ("Col. 5" in col or "Col.5" in col or "Col. (5)" in col):
            return col
    for col in df.columns:
        if "Total" in col and ("Col. 6" in col or "Col.6" in col or "Col. (6)" in col):
            return col
    for col in df.columns:
        if "Total" in col and "Number" in col:
            return col
    for col in df.columns:
        if "Total" in col and "Death" in col:
            return col
    return None

for year in YEARS:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()

    cat_col, name_col = detect_cols(df)

    df = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]

    total_col = find_total_col(df, year)

    if total_col is None:
        print(f"WARNING: Total column not found for year {year}")
        continue

    df_clean = df[[name_col, total_col]].copy()
    df_clean.columns = ["State/UT", "Total Accidental Deaths"]

    df_clean["State/UT"] = df_clean["State/UT"].astype(str).str.strip().str.upper().map(STATE_UT_NAMES)
    df_clean = df_clean.dropna(subset=["State/UT"])
    df_clean["Year"] = year
    df_clean["Total Accidental Deaths"] = pd.to_numeric(df_clean["Total Accidental Deaths"], errors="coerce")
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False)["Total Accidental Deaths"].sum()

    all_years_data.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 35 States/UTs processed.
2016 — 35 States/UTs processed.
2017 — 35 States/UTs processed.
2018 — 35 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 35 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 35 States/UTs processed.

All years processed!


In [146]:
# Merge all years and add S.No.
final_df = pd.concat(all_years_data, ignore_index=True)

# Sort by Year then alphabetically by State/UT
final_df = final_df.sort_values(["Year", "State/UT"]).reset_index(drop=True)

# Add continuous S.No.
final_df.insert(0, "S.No.", range(1, len(final_df) + 1))

# Reorder columns
final_df = final_df[["S.No.", "State/UT", "Year", "Total Accidental Deaths"]]

# Save to processed folder
final_df.to_csv("../data/processed/total_accidental_deaths_2015_2022.csv", index=False)

print(f"Total rows: {len(final_df)}")
print(f"Years: {final_df['Year'].unique()}")
print(f"\nFirst 10 rows:")
print(final_df.head(10))

Total rows: 280
Years: [2015 2016 2017 2018 2019 2020 2021 2022]

First 10 rows:
   S.No.                                  State/UT  Year  \
0      1               Andaman and Nicobar Islands  2015   
1      2                            Andhra Pradesh  2015   
2      3                         Arunachal Pradesh  2015   
3      4                                     Assam  2015   
4      5                                     Bihar  2015   
5      6                                Chandigarh  2015   
6      7                              Chhattisgarh  2015   
7      8  Dadra and Nagar Haveli and Daman and Diu  2015   
8      9                                     Delhi  2015   
9     10                                       Goa  2015   

   Total Accidental Deaths  
0                       96  
1                    15320  
2                      319  
3                     3910  
4                     9470  
5                      199  
6                    14333  
7                      227

In [147]:
# Debug 2022 Table_1A.6 Dadra raw name
df_2022 = all_files[2022]["Table_1A.6"]["df"].copy()
df_2022.columns = df_2022.columns.str.strip()
name_col = df_2022.columns[1]
dadra = [n for n in df_2022[name_col].astype(str).str.strip().tolist() if "DADRA" in n.upper() or "DAMAN" in n.upper() or "HAVELI" in n.upper()]
print(f"2022 Dadra raw name: {dadra}")

2022 Dadra raw name: ['Andaman and Nicobar Islands', 'Dadra and Nicobar Haveli and Daman and Diu']


In [148]:
# Fix - add typo to STATE_UT_NAMES and update VALID_STATE_UT_KEYS
STATE_UT_NAMES["DADRA AND NICOBAR HAVELI AND DAMAN AND DIU"] = "Dadra and Nagar Haveli and Daman and Diu"
VALID_STATE_UT_KEYS = set(STATE_UT_NAMES.keys())
print("Fixed!")

Fixed!


In [149]:
# Fix mapping
STATE_UT_NAMES["DADRA AND NICOBAR HAVELI AND DAMAN AND DIU"] = "Dadra and Nagar Haveli and Daman and Diu"
VALID_STATE_UT_KEYS = set(STATE_UT_NAMES.keys())

# Clean Table_1A.6 - Time of Occurrence - FIXED
all_years_time = []

TIME_MAPPING_FIXED = {
    "0000 hrs to 0300 hrs (Night)": [
        "Road Accidents - 0000 hrs to 0300 hrs. (Night)",
        "Road Accidents_0000 hrs to 0300 hrs. (Night)"
    ],
    "0300 hrs to 0600 hrs (Night)": [
        "Road Accidents - 0300 hrs to 0600 hrs. (Night)",
        "Road Accidents_0300 hrs to 0600 hrs. (Night)"
    ],
    "0600 hrs to 0900 hrs (Day)": [
        "Road Accidents - 0600 hrs to 0900 hrs (Day)",
        "Road Accidents - 0600 hrs to 0900 hrs. (Day)",
        "Road Accidents_0600 hrs to 0900 hrs. (Day)"
    ],
    "0900 hrs to 1200 hrs (Day)": [
        "Road Accidents - 0900 hrs to 1200 hrs (Day)",
        "Road Accidents - 0900 hrs to 1200 hrs. (Day)",
        "Road Accidents_0900 hrs to 1200 hrs. (Day)"
    ],
    "1200 hrs to 1500 hrs (Day)": [
        "Road Accidents - 1200 hrs to 1500 hrs (Day)",
        "Road Accidents - 1200 hrs to 1500 hrs. (Day)",
        "Road Accidents_1200 hrs to 1500 hrs. (Day)"
    ],
    "1500 hrs to 1800 hrs (Day)": [
        "Road Accidents - 1500 hrs to 1800 hrs (Day)",
        "Road Accidents - 1500 hrs to 1800 hrs. (Day)",
        "Road Accidents_1500 hrs to 1800 hrs. (Day)"
    ],
    "1800 hrs to 2100 hrs (Night)": [
        "Road Accidents - 1800 hrs to 2100 hrs (Night)",
        "Road Accidents - 1800 hrs to 2100 hrs. (Night)",
        "Road Accidents - 1800 hrs to 2100hrs (Night)",
        "Road Accidents_1800 hrs to 2100 hrs. (Night)"
    ],
    "2100 hrs to 2400 hrs (Night)": [
        "Road Accidents - 2100 hrs to 2400 hrs (Night)",
        "Road Accidents - 2100 hrs to 2400 hrs. (Night)",
        "Road Accidents - 2100 hrs. to 2400hrs (Night)",
        "Road Accidents - 2100 hrs. to 2400hrs(Night)",
        "Road Accidents_2100 hrs to 2400 hrs. (Night)"
    ],
}

for year in YEARS:
    df = all_files[year]["Table_1A.6"]["df"].copy()
    df.columns = df.columns.str.strip()

    cat_col, name_col = detect_transport_cols(df, year)

    if cat_col is not None:
        df = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    else:
        df = df[df[name_col].astype(str).str.strip().str.upper().isin(VALID_STATE_UT_KEYS)]

    df = df[~df[name_col].astype(str).str.strip().str.upper().str.startswith("TOTAL")]

    df_clean = df[[name_col]].copy()
    df_clean.columns = ["State/UT"]

    for new_col, source_cols in TIME_MAPPING_FIXED.items():
        available = [c for c in source_cols if c in df.columns]
        if available:
            df_clean[new_col] = df[available[0]].apply(pd.to_numeric, errors="coerce")
        else:
            df_clean[new_col] = 0
            print(f"WARNING: {year} — No source column found for {new_col}")

    df_clean["State/UT"] = df_clean["State/UT"].astype(str).str.strip().str.upper().map(STATE_UT_NAMES)
    df_clean = df_clean.dropna(subset=["State/UT"])
    df_clean["Year"] = year
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False).sum()

    all_years_time.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

# Save
final_time_df = pd.concat(all_years_time, ignore_index=True)
final_time_df = final_time_df.sort_values(["Year", "State/UT"]).reset_index(drop=True)
final_time_df.insert(0, "S.No.", range(1, len(final_time_df) + 1))
final_time_df = final_time_df[["S.No.", "State/UT", "Year",
                                "0000 hrs to 0300 hrs (Night)",
                                "0300 hrs to 0600 hrs (Night)",
                                "0600 hrs to 0900 hrs (Day)",
                                "0900 hrs to 1200 hrs (Day)",
                                "1200 hrs to 1500 hrs (Day)",
                                "1500 hrs to 1800 hrs (Day)",
                                "1800 hrs to 2100 hrs (Night)",
                                "2100 hrs to 2400 hrs (Night)"]]
final_time_df.to_csv("../data/processed/time_of_occurrence_2015_2022.csv", index=False)
print(f"\nTotal rows: {len(final_time_df)}")

2015 — 35 States/UTs processed.
2016 — 35 States/UTs processed.
2017 — 35 States/UTs processed.
2018 — 35 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 35 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 35 States/UTs processed.

Total rows: 280


In [150]:
# Fix STATE_UT_NAMES - add all typos permanently
STATE_UT_NAMES["DADRA AND NICOBAR HAVELI AND DAMAN AND DIU"] = "Dadra and Nagar Haveli and Daman and Diu"
VALID_STATE_UT_KEYS = set(STATE_UT_NAMES.keys())
print("Mapping permanently fixed!")

Mapping permanently fixed!


In [151]:
# Check Dadra typo in all processed files
import pandas as pd
import os

files = {
    "deaths": "../data/processed/total_accidental_deaths_2015_2022.csv",
    "transport": "../data/processed/mode_of_transport_2015_2022.csv",
    "month": "../data/processed/month_of_occurrence_2015_2022.csv",
    "time": "../data/processed/time_of_occurrence_2015_2022.csv",
    "road": "../data/processed/road_classification_2015_2022.csv",
    "place": "../data/processed/place_of_occurrence_2015_2022.csv",
}

for name, path in files.items():
    df = pd.read_csv(path)
    print(f"{name}: {len(df)} rows")

deaths: 280 rows
transport: 280 rows
month: 280 rows
time: 280 rows
road: 280 rows
place: 280 rows
